In [ ]:
import os
import glob
import re
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import pickle
import xgboost as xgb
import gc

from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier, plot_importance
from collections import defaultdict
from typing import Optional, Sequence, Tuple, List, Union

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['mathtext.rm'] = 'serif'
plt.style.use(hep.style.CMS)

In [ ]:
# --- Configuration ---
BASE_DIR     = "/afs/ihep.ac.cn/users/y/yiyangzhao/Research/CMS_THU_Space/VVV/train/dataset"
SIG_DIR      = os.path.join(BASE_DIR, "signal")
BKG_DIR      = os.path.join(BASE_DIR, "bkg")
DATA_DIR     = os.path.join(BASE_DIR, "data")
RANDOM_STATE = 42
ENTRIES_PER_SAMPLE = 1000000
DECOR_LAMBDA = 20          # 去相关惩罚强度（如需更强平坦化可调大一些，建议 0.5~5 扫描）
_EPS = 1e-12                # 数值稳定项

In [ ]:
global SCALE_2FAT
SCALE_2FAT = 0
global SCALE_3FAT
SCALE_3FAT = 0

global SCALE_2FAT_VVV
SCALE_2FAT_VVV = 0
global SCALE_2FAT_VH
SCALE_2FAT_VH = 0
global SCALE_2FAT_VV
SCALE_2FAT_VV = 0
global SCALE_2FAT_TT
SCALE_2FAT_TT = 0
global SCALE_2FAT_QCD
SCALE_2FAT_QCD = 0
global SCALE_3FAT_VVV
SCALE_3FAT_VVV = 0
global SCALE_3FAT_VH
SCALE_3FAT_VH = 0
global SCALE_3FAT_VV
SCALE_3FAT_VV = 0
global SCALE_3FAT_TT
SCALE_3FAT_TT = 0
global SCALE_3FAT_QCQ
SCALE_3FAT_QCD = 0

In [ ]:
branches2 =  [
    "N_ak8", "N_ak4", "H_T",
    #"N_PV", "N_e", "N_mu", "N_gamma", 
    "pt8_1", "pt8_2", "eta8_1", "eta8_2", "msd8_1", "msd8_2", "mr8_1", "mr8_2",
    #"nConst8_1", "nConst8_2", "nCh8_1", "nCh8_2", "nEle8_1", "nEle8_2",
    #"nMu8_1", "nMu8_2", "nNh8_1", "nNh8_2", "nPho8_1", "nPho8_2",
    #"area8_1", "area8_2", "chEmEF8_1", "chEmEF8_2", "chHEF8_1", "chHEF8_2",
    #"hfEmEF8_1", "hfEmEF8_2", "hfHEF8_1", "hfHEF8_2", "neEmEF8_1", "neEmEF8_2",
    #"neHEF8_1", "neHEF8_2", "muEF8_1", "muEF8_2", "n2b1_1", "n2b1_2",
    #"n3b1_1", "n3b1_2", "tau21_1", "tau21_2", "tau32_1", "tau32_2",
    "WvsQCD_1", "WvsQCD_2",
    "pt4_1", "pt4_2", "pt4_3", "pt4_4", "eta4_1", "eta4_2", "eta4_3", "eta4_4",
    "mPF4_1", "mPF4_2", "mPF4_3", "mPF4_4", "nConst4_1", "nConst4_2",
    "nConst4_3", "nConst4_4", "nCh4_1", "nCh4_2", "nCh4_3", "nCh4_4",
    "nEle4_1", "nEle4_2", "nEle4_3", "nEle4_4",
    #"nMu4_1", "nMu4_2", "nMu4_3", "nMu4_4", 
    #"nNh4_1", "nNh4_2", "nNh4_3", "nNh4_4", "nPho4_1", "nPho4_2", "nPho4_3", "nPho4_4", 
    "area4_1", "area4_2", "area4_3", "area4_4", 
    "chEmEF4_1", "chEmEF4_2", "chEmEF4_3", "chEmEF4_4",
    "chHEF4_1", "chHEF4_2", "chHEF4_3", "chHEF4_4", 
    #"hfEmEF4_1", "hfEmEF4_2", "hfEmEF4_3", "hfEmEF4_4", "hfHEF4_1", "hfHEF4_2", "hfHEF4_3", "hfHEF4_4",
    "neEmEF4_1", "neEmEF4_2", "neEmEF4_3", "neEmEF4_4", "neHEF4_1", "neHEF4_2",
    "neHEF4_3", "neHEF4_4", 
    #"muEF4_1", "muEF4_2", "muEF4_3", "muEF4_4",
    "PT", "dR8", "dPhi", "m1overM", "m2overM","sphereM",
    #"M8", "M84", "pt1overPT", "pt2overPT", "PToverptsum", 
    "dR84_min", "dR44_min", "dR8L_min",
    "ptL_1", "ptL_2", "ptL_3", 
    "etaL_1", "etaL_2", "etaL_3", "phiL_1", "phiL_2", "phiL_3", 
    "isoEcalL_1", "isoEcalL_2", "isoEcalL_3", "isoHcalL_1", "isoHcalL_2", "isoHcalL_3"
]

In [ ]:
branches3 = [
    "N_ak8", "N_ak4", "H_T",
    #"N_PV", "N_e", "N_mu", "N_gamma", 
    "pt8_1", "pt8_2", "pt8_3", "eta8_1", "eta8_2", "eta8_3",
    "msd8_1", "msd8_2", "msd8_3", "mr8_1", "mr8_2", "mr8_3",
    #"nConst8_1", "nConst8_2", "nConst8_3", "nCh8_1", "nCh8_2", "nCh8_3",
    #"nEle8_1", "nEle8_2", "nEle8_3", "nMu8_1", "nMu8_2", "nMu8_3",
    #"nNh8_1", "nNh8_2", "nNh8_3", "nPho8_1", "nPho8_2", "nPho8_3",
    #"area8_1", "area8_2", "area8_3", "chEmEF8_1", "chEmEF8_2", "chEmEF8_3",
    #"chHEF8_1", "chHEF8_2", "chHEF8_3", "hfEmEF8_1", "hfEmEF8_2", "hfEmEF8_3",
    #"hfHEF8_1", "hfHEF8_2", "hfHEF8_3", "neEmEF8_1", "neEmEF8_2", "neEmEF8_3",
    #"neHEF8_1", "neHEF8_2", "neHEF8_3", "muEF8_1", "muEF8_2", "muEF8_3",
    #"n2b1_1", "n2b1_2", "n2b1_3", "n3b1_1", "n3b1_2", "n3b1_3",
    #"tau21_1", "tau21_2", "tau21_3", "tau32_1", "tau32_2", "tau32_3",
    "WvsQCD_1", "WvsQCD_2", "WvsQCD_3",
    "sphereM", "M", "m1overM", "m2overM", "m3overM", 
    "PT",
    #"pt1overPT", "pt2overPT", "pt3overPT", "PToverptsum", 
    "dR_min", "dR_max", "dPhi_min", "dPhi_max", "dRL_min",
    "ptL_1", "ptL_2", "ptL_3", 
    "etaL_1", "etaL_2", "etaL_3", "phiL_1", "phiL_2", "phiL_3", 
    "isoEcalL_1", "isoEcalL_2", "isoEcalL_3",
    "isoHcalL_1", "isoHcalL_2", "isoHcalL_3"
]

In [ ]:
BRANCH_CLIP_RANGES = {
    "H_T":   (0, 13600),
    "M":     (0, 13600),
    "M8":    (0, 13600),
    "M84":   (0, 13600),
    "PT":    (0, 13600),
    "PT":    (0, 13600),
    "pt4_1": (0, 13600),
    "pt4_2": (0, 13600),
    "pt4_3": (0, 13600),
    "pt4_4": (0, 13600),
    "pt8_1": (0, 13600),
    "pt8_2": (0, 13600),
    "pt8_3": (0, 13600),
    "ptL_1": (0, 13600),
    "ptL_2": (0, 13600),
    "ptL_3": (0, 13600)
}

In [ ]:
# utility: group files by prefix without trailing _1, _2

def group_files(path):
    files = glob.glob(os.path.join(path, "*.root"))
    groups = {}
    for f in files:
        name = os.path.splitext(os.path.basename(f))[0]
        key  = re.sub(r'_[0-9]+$', '', name)
        if '202' in key:
            key = 'data'
        groups.setdefault(key, []).append(f)
    return groups

In [ ]:
# x-section [pb] / raw entries * 1e8
_RAW_XSEC = {
    "www":              4.852,
    "wwz":              1.024,
    "wzz":              0.344,
    "zzz":              0.0863,
    "ww":               118.9,
    "wz":               45.16,
    "zz":               253.5,
    "qcd_ht100to200":   1.973e7,
    "qcd_ht200to400":   1.703e6,
    "qcd_ht400to600":   8.970e4,
    "qcd_ht600to800":   1.056e4,
    "qcd_ht800to1000":  2413,
    "qcd_ht1000to1200": 772.4,
    "qcd_ht1200to1500": 356.2,
    "qcd_ht1500to2000": 110.7,
    "qcd_ht2000toinf":  29.56,
    "tt_had":           79.80,
    "tt_semilep":       75.68,
    "zh":           2.26,
    "wplush":       3.72,
    "wminush":      2.15,
    "data":         20821
}

WEIGHT_2FAT = {}
WEIGHT_3FAT = {}

_RAW_XSEC_fat2 = {}
_RAW_XSEC_fat3 = {}
XSEC_MAP = {
    "fat2": _RAW_XSEC_fat2,
    "fat3": _RAW_XSEC_fat3,
}


RAW_XSEC = {
    k.lower().replace(' ', '_'): v
    for k, v in _RAW_XSEC.items()
}

In [ ]:
def count_entries(path, tree_name):
    counts = {}
    groups = group_files(path)
    #print(groups)
    for sample, files in groups.items():
        total = 0
        for fpath in files:
            with uproot.open(fpath) as uf:
                if tree_name in uf:
                    tree = uf[tree_name]
                    total += tree.num_entries
        counts[sample] = total
    return counts

# 分别统计 signal/ bkg 下 fat2, fat3 的 entry
sig_fat2 = count_entries(SIG_DIR, "fat2")
sig_fat3 = count_entries(SIG_DIR, "fat3")
bkg_fat2 = count_entries(BKG_DIR, "fat2")
bkg_fat3 = count_entries(BKG_DIR, "fat3")
data_fat2 = count_entries(DATA_DIR, "fat2")
data_fat3 = count_entries(DATA_DIR, "fat3")

for sample, xsec in _RAW_XSEC.items():
    e2 = sig_fat2.get(sample, 0) + bkg_fat2.get(sample, 0) + data_fat2.get(sample, 0)
    e3 = sig_fat3.get(sample, 0) + bkg_fat3.get(sample, 0) + data_fat3.get(sample, 0)
    _RAW_XSEC_fat2[sample] = xsec * e2
    _RAW_XSEC_fat3[sample] = xsec * e3
    #print(f"{sample}: {e2} {e3}")

In [ ]:
def filter_X(X: pd.DataFrame, y, w, branch: list, thresholds: dict = None, apply_to_sentinel: bool = True):
    """
    依据 thresholds 对指定列进行筛选，并处理哨兵值（< -990）。
    新增：thresholds 的每个键可支持“分多段的 & 或 |”的复合条件。

    条件 cond 的支持形式（对单列）：
      - 标量 c                 ->  col > c
      - 二元组 (mn, mx)        ->  (mn is None 或 col > mn) 且 (mx is None 或 col < mx)
      - 列表 [cond1, cond2...] ->  视作 OR：cond1 | cond2 | ...
      - 字典 {'|': [..]}       ->  多段 OR
      - 字典 {'&': [..]}       ->  多段 AND
    以上可递归嵌套，例如：{'&': [ (None,200), {'|':[ (0,40), (150,None) ]} ]}

    apply_to_sentinel=True 时：先剔除哨兵，再按数值条件筛选（若有条件）。
    apply_to_sentinel=False 时：哨兵行自动通过，其余行按数值条件筛选（若有条件）。
    """
    if thresholds is None:
        return X.copy(), y.copy(), w.copy()

    # 从全 True 开始，逐列收紧
    mask = pd.Series(True, index=X.index)

    def _combine_masks(masks: list[pd.Series], op: str, idx) -> pd.Series:
        if not masks:
            # 空集合在 AND 中返回全 True，在 OR 中返回全 False
            return pd.Series(True, index=idx) if op == '&' else pd.Series(False, index=idx)
        out = masks[0]
        for m in masks[1:]:
            out = (out & m) if op == '&' else (out | m)
        return out

    def _mask_from_cond(col: pd.Series, cond) -> pd.Series:
        """将任意支持的 cond 转成布尔掩码；不处理 sentinel。"""
        idx = col.index

        # None -> 全 True（相当于无条件）
        if cond is None:
            return pd.Series(True, index=idx)

        # 数值标量：col > c
        if isinstance(cond, (int, float, np.integer, np.floating)):
            return col > float(cond)

        # 二元组 (mn, mx)
        if isinstance(cond, tuple) and len(cond) == 2 and not isinstance(cond[0], (list, dict, tuple)):
            mn, mx = cond
            m = pd.Series(True, index=idx)
            if mn is not None:
                m &= (col > mn)
            if mx is not None:
                m &= (col < mx)
            return m

        # 列表：默认 OR
        if isinstance(cond, (list, tuple)):
            masks = [_mask_from_cond(col, c) for c in cond]
            return _combine_masks(masks, '|', idx)

        # 字典：{'|': [..]} 或 {'&': [..]}，支持递归
        if isinstance(cond, dict):
            if '|' in cond:
                masks = [_mask_from_cond(col, c) for c in cond['|']]
                return _combine_masks(masks, '|', idx)
            if '&' in cond:
                masks = [_mask_from_cond(col, c) for c in cond['&']]
                return _combine_masks(masks, '&', idx)
            # 兼容小写字符串键
            if 'or' in cond:
                masks = [_mask_from_cond(col, c) for c in cond['or']]
                return _combine_masks(masks, '|', idx)
            if 'and' in cond:
                masks = [_mask_from_cond(col, c) for c in cond['and']]
                return _combine_masks(masks, '&', idx)
            raise ValueError(f"Unsupported dict condition keys for column: {cond}")

        raise TypeError(f"Unsupported condition type for column: {type(cond)}")

    for b in branch:
        if b not in X.columns:
            raise KeyError(f"Column {b!r} not found in X")

        col = X[b]
        cond = thresholds.get(b, None)
        sentinel = col < -990  # True 表示哨兵值行

        if apply_to_sentinel:
            # 先剔除所有哨兵行
            mask &= ~sentinel
            # 再对剩下的行应用数值阈值
            if cond is not None:
                cond_mask = _mask_from_cond(col, cond)
                mask &= cond_mask
        else:
            # 哨兵行自动通过，只有非哨兵才做阈值判断
            if cond is not None:
                cond_mask = _mask_from_cond(col, cond)
                mask &= (cond_mask | sentinel)

    return X.loc[mask].copy(), y[mask.values].copy(), w[mask.values].copy()


In [ ]:
def standardize_X(X: pd.DataFrame) -> pd.DataFrame:
    # 需要做 ln 变换（b）的列名规则（区分大小写）
    def _need_ln(col: str) -> bool:
        if col.startswith("N_"):
            return True
        if col == "H_T":
            return True
        if col.startswith("pt"):
            return True
        if col.startswith("sphereM"):
            return True
        if col.startswith("iso"):
            return True
        if col.startswith("M"):
            return True
        if col == "PT":
            return True
        if col.startswith("mPF"):
            return True
        if col.startswith(("nMu", "nNh", "nPho", "nCh", "nEle", "nConst")):
            return True
        if "overPT" in col:
            return True
        if col.startswith("mr"):
            return True
        return False

    for col in X.columns:
        arr = X[col].values  # 底层 numpy 视图（通常是 float；若不是，下面会处理）

        # 1) 哨兵值：-999（这里沿用原逻辑：< -990 都视为哨兵），不改变
        mask = arr < -990
        valid = ~mask
        if not valid.any():
            continue

        # ---- 2) 进行裁切（仅对非哨兵值） --------------------------------------
        clip_min, clip_max = BRANCH_CLIP_RANGES.get(col, (None, None))
        if clip_min is not None:
            arr[valid & (arr < clip_min)] = clip_min
        if clip_max is not None:
            arr[valid & (arr > clip_max)] = clip_max

        # ---- 3) 不再做均值方差标准化：只按规则选择是否做 ln 变换 --------------
        if _need_ln(col):
            # 为了避免 ln(0) -> -inf，这里只对 >0 的有效值取 ln；<=0 的有效值保持不变
            pos = valid & (arr > 0)
            if pos.any():
                # 若列是整型，这里会产生浮点结果；确保写回不会被截断
                if not np.issubdtype(arr.dtype, np.floating):
                    arr = arr.astype(float, copy=True)
                arr[pos] = np.log(arr[pos])

        X[col] = arr

    return X


# Multi-class

In [ ]:
TYPE = {
    "www":              0,
    "wwz":              1,
    "wzz":              2,
    "zzz":              3,
    "zh":               4,
    "wplush":           5,
    "wminush":          5,
    "qcd_ht100to200":   10,
    "qcd_ht200to400":   10,
    "qcd_ht400to600":   10,
    "qcd_ht600to800":   10,
    "qcd_ht800to1000":  10,
    "qcd_ht1000to1200": 10,
    "qcd_ht1200to1500": 10,
    "qcd_ht1500to2000": 10,
    "qcd_ht2000toinf":  10,
    "tt_had":           11,
    "tt_semilep":       12,
    "ww":               13,
    "wz":               14,
    "zz":               15,
    "data":             -2,
    "unknown":          -1,
}

In [ ]:
def _map_to_3classes(y_raw: np.ndarray) -> np.ndarray:
    y_raw = np.asarray(y_raw)
    y = np.full_like(y_raw, fill_value=-999, dtype=int)

    # VVV 类
    vvv_codes = {TYPE["www"], TYPE["wwz"], TYPE["wzz"], TYPE["zzz"]}
    # VH  类
    vh_codes  = {TYPE["zh"], TYPE["wplush"], TYPE["wminush"]}
    # TTbar 类
    tt_codes  = {TYPE["tt_had"], TYPE["tt_semilep"]}
    # VV 类
    vv_codes  = {TYPE["ww"], TYPE["wz"], TYPE["zz"]}
    # QCD 类
    qcd_codes = {TYPE["qcd_ht100to200"], TYPE["qcd_ht200to400"], TYPE["qcd_ht400to600"], TYPE["qcd_ht600to800"], TYPE["qcd_ht800to1000"], TYPE["qcd_ht1000to1200"], TYPE["qcd_ht1200to1500"], TYPE["qcd_ht1500to2000"], TYPE["qcd_ht2000toinf"]}
    # 排除
    exclude   = {TYPE["data"], TYPE["unknown"]}

    # 标注
    y[np.isin(y_raw, list(vvv_codes))] = 0
    y[np.isin(y_raw, list(vh_codes))]  = 1
    y[np.isin(y_raw, list(tt_codes))] = 2
    y[np.isin(y_raw, list(vv_codes))]  = 3
    y[np.isin(y_raw, list(qcd_codes))]  = 4
    # 背景（不是排除，并且不是上面两类的，均归为 BKG=0）
    mask_valid_not_set = (~np.isin(y_raw, list(exclude))) & (y == -999)
    y[mask_valid_not_set] = -999

    return y, ~np.isin(y_raw, list(exclude))

def _smooth_boost(prob: np.ndarray, floor: float, bmin: float, bmax, power: float) -> np.ndarray:
    prob = np.asarray(prob)
    bmax_arr = np.asarray(bmax)
    eps = 1e-8
    denom = np.maximum(1.0 - floor, eps)
    t = np.clip((prob - floor) / denom, 0.0, 1.0) ** power
    return bmin + (bmax_arr - bmin) * t

In [ ]:
def prepare_multi_data(tree_name, branches, entries_per_sample,
                 SIG_DIR=SIG_DIR, BKG_DIR=BKG_DIR):

    def load_samples(path):
        """
        按 sample 分组读取文件，每个 sample 最多取 limit_per_sample 条目，
        分配均匀的 per-event weight，使得 sum(weight_i)=xsec(sample)。
        """
        groups = group_files(path)  # { sample_id: [file1.root, file2.root, ...] }
        dfs = []
        
        for sample_id, file_list in groups.items():
            sid = sample_id.lower()
            limit = entries_per_sample
            if 'data' in sample_id:
            #    #sample_id = 'data'
            #    sid = 'data'
                limit = 1e4
            #    #print(limit)
            if sid not in XSEC_MAP[tree_name]:
                raise KeyError(f"Sample '{sample_id}' not found in RAW_XSEC map")
            xsec = XSEC_MAP[tree_name][sample_id]
            
            parts = []
            n_read = 0
            for fpath in file_list:
                remain = limit - n_read
                if remain <= 0:
                    break

                with uproot.open(fpath) as uf:
                    tree = uf[tree_name]
                    # 本文件总条目数
                    n_entries = tree.num_entries
                    # 本次从该文件中要读取的条目数
                    n_to_read = min(remain, n_entries)
                    if n_to_read <= 0:
                        continue

                    # 只读取前 n_to_read 条
                    df_part = tree.arrays(
                        branches + ["type"],
                        library='pd',
                        entry_start=0,
                        entry_stop=n_to_read
                    )

                parts.append(df_part)
                n_read += len(df_part)
                if n_read >= limit:
                    break
            
            if n_read == 0:
                continue
            
            df_sample = pd.concat(parts, ignore_index=True)
            del parts
            gc.collect()
            # 2. 均匀分配 weight 使 sum_i w_i == xsec
            per_evt_w = xsec / float(n_read)
            df_sample["weight"]   = per_evt_w

            df_sample["type"] = int(TYPE[sample_id])
            type = df_sample["type"].values[0]
            
            # 4. 每个 sample 内部打乱、重置索引
            #df_sample = df_sample.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
            dfs.append(df_sample)
            print(f"Sample '{sample_id}': read {n_read} entries (limit {limit}) with weight {per_evt_w} Type: {type}")
            if (tree_name == "fat2"):
                WEIGHT_2FAT[sample_id] = per_evt_w
            elif (tree_name == "fat3"):
                WEIGHT_3FAT[sample_id] = per_evt_w
        
        if dfs:
            return pd.concat(dfs, ignore_index=True)
        else:
            # 万一某类全没读到，返回空 DataFrame
            cols = branches + ["weight", "type"]
            return pd.DataFrame(columns=cols)
    
    # 读取 signal、background
    df_sig = load_samples(SIG_DIR)
    df_bkg = load_samples(BKG_DIR)
    df_data = load_samples(DATA_DIR)

    # 3. 缩放总权重 = 背景总权重，对于TYPE中的VVV、VH分别计算scale
    w_sig_vvv = df_sig[df_sig["type"].isin([TYPE["www"], TYPE["wwz"], TYPE["wzz"], TYPE["zzz"]])]["weight"].sum()
    w_sig_vh  = df_sig[df_sig["type"].isin([TYPE["zh"], TYPE["wplush"], TYPE["wminush"]])]["weight"].sum()
    w_bkg_vv  = df_bkg[df_bkg["type"].isin([TYPE["ww"], TYPE["wz"], TYPE["zz"]])]["weight"].sum()
    w_bkg_tt  = df_bkg[df_bkg["type"].isin([TYPE["tt_had"], TYPE["tt_semilep"]])]["weight"].sum()
    w_bkg_qcd = df_bkg[df_bkg["type"].isin([TYPE["qcd_ht100to200"], TYPE["qcd_ht200to400"], TYPE["qcd_ht400to600"], TYPE["qcd_ht600to800"], TYPE["qcd_ht800to1000"], TYPE["qcd_ht1000to1200"], TYPE["qcd_ht1200to1500"], TYPE["qcd_ht1500to2000"], TYPE["qcd_ht2000toinf"]])]["weight"].sum()
    
    n_sig_vvv = len(df_sig[df_sig["type"].isin([TYPE["www"], TYPE["wwz"], TYPE["wzz"], TYPE["zzz"]])])
    n_sig_vh  = len(df_sig[df_sig["type"].isin([TYPE["zh"], TYPE["wplush"], TYPE["wminush"]])])
    n_bkg_vv  = len(df_bkg[df_bkg["type"].isin([TYPE["ww"], TYPE["wz"], TYPE["zz"]])])
    n_bkg_tt  = len(df_bkg[df_bkg["type"].isin([TYPE["tt_had"], TYPE["tt_semilep"]])])
    n_bkg_qcd = len(df_bkg[df_bkg["type"].isin([TYPE["qcd_ht100to200"], TYPE["qcd_ht200to400"], TYPE["qcd_ht400to600"], TYPE["qcd_ht600to800"], TYPE["qcd_ht800to1000"], TYPE["qcd_ht1000to1200"], TYPE["qcd_ht1200to1500"], TYPE["qcd_ht1500to2000"], TYPE["qcd_ht2000toinf"]])])
    
    if w_sig_vvv > 0 and w_sig_vh > 0:
        scale_vvv  = 1e10 / (w_sig_vvv)
        scale_vh   = 1e10 / (w_sig_vh)
        scale_vv   = 1e10 / (w_bkg_vv)
        scale_tt   = 1e10 / (w_bkg_tt)
        scale_qcd  = 1e10 / (w_bkg_qcd)
        
        df_sig.loc[df_sig["type"].isin([TYPE["www"], TYPE["wwz"], TYPE["wzz"], TYPE["zzz"]]), "weight"] = df_sig.loc[df_sig["type"].isin([TYPE["www"], TYPE["wwz"], TYPE["wzz"], TYPE["zzz"]]), "weight"] * scale_vvv
        df_sig.loc[df_sig["type"].isin([TYPE["zh"], TYPE["wplush"], TYPE["wminush"]]), "weight"] = df_sig.loc[df_sig["type"].isin([TYPE["zh"], TYPE["wplush"], TYPE["wminush"]]), "weight"] * scale_vh
        df_bkg.loc[df_bkg["type"].isin([TYPE["ww"], TYPE["wz"], TYPE["zz"]]), "weight"] = df_bkg.loc[df_bkg["type"].isin([TYPE["ww"], TYPE["wz"], TYPE["zz"]]), "weight"] * scale_vv
        df_bkg.loc[df_bkg["type"].isin([TYPE["tt_had"], TYPE["tt_semilep"]]), "weight"] = df_bkg.loc[df_bkg["type"].isin([TYPE["tt_had"], TYPE["tt_semilep"]]), "weight"] * scale_tt
        df_bkg.loc[df_bkg["type"].isin([TYPE["qcd_ht100to200"], TYPE["qcd_ht200to400"], TYPE["qcd_ht400to600"], TYPE["qcd_ht600to800"], TYPE["qcd_ht800to1000"], TYPE["qcd_ht1000to1200"], TYPE["qcd_ht1200to1500"], TYPE["qcd_ht1500to2000"], TYPE["qcd_ht2000toinf"]]), "weight"] = df_bkg.loc[df_bkg["type"].isin([TYPE["qcd_ht100to200"], TYPE["qcd_ht200to400"], TYPE["qcd_ht400to600"], TYPE["qcd_ht600to800"], TYPE["qcd_ht800to1000"], TYPE["qcd_ht1000to1200"], TYPE["qcd_ht1200to1500"], TYPE["qcd_ht1500to2000"], TYPE["qcd_ht2000toinf"]]), "weight"] * scale_qcd

        if tree_name == "fat2": 
            SCALE_2FAT_VVV = scale_vvv
            SCALE_2FAT_VH  = scale_vh
            SCALE_2FAT_VV = scale_vv
            SCALE_2FAT_TT = scale_tt
            SCALE_2FAT_QCD = scale_qcd
            print(f"SCALE_2FAT_VVV: {scale_vvv}")
            print(f"SCALE_2FAT_VH: {scale_vh}")
            print(f"SCALE_2FAT_VV: {scale_vv}")
            print(f"SCALE_2FAT_TT: {scale_tt}")
            print(f"SCALE_2FAT_QCD: {scale_qcd}")
        elif tree_name == "fat3": 
            SCALE_3FAT_VVV = scale_vvv
            SCALE_3FAT_VH  = scale_vh
            SCALE_3FAT_VV = scale_vv
            SCALE_3FAT_TT = scale_tt
            SCALE_3FAT_QCD = scale_qcd
            print(f"SCALE_3FAT_VVV: {scale_vvv}")
            print(f"SCALE_3FAT_VH: {scale_vh}")
            print(f"SCALE_3FAT_VV: {scale_vv}")
            print(f"SCALE_3FAT_TT: {scale_tt}")
            print(f"SCALE_3FAT_QCD: {scale_qcd}")

    # 4. 合并后全体再打乱、重置索引
    df_all = pd.concat([df_sig, df_bkg], ignore_index=True)
    del df_sig, df_bkg
    gc.collect()
    
    df_all = df_all.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    # 5. 拆分标签、权重、特征
    X_data = df_data[branches]
    y_data = df_data.pop("type").values
    w_data = df_data.pop("weight").values
    del df_data
    gc.collect()
    
    X = df_all[branches]
    y = df_all.pop("type").values
    w = df_all.pop("weight").values
    
    del df_all
    gc.collect()

    return X, y, w, X_data, y_data, w_data

In [ ]:
def train_multi_model(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    model_name: str,
):
    """
    多分类（BKG/VVV/VH）训练，二次训练时对 “VVV 分数高” 与 “VH 分数高”的样本同时放大权重（两侧放大系数相乘）。
    其他流程、参数风格与原实现保持一致。
    返回：
      clf, (X_train, X_test, y_train_3c, y_test_3c, w_train, w_test)
    """
    X = np.asarray(X)
    y = np.asarray(y)
    w = np.asarray(w, dtype=float)

    # 0) 将原始标签映射为 3 类，并过滤掉 data/unknown
    y5, valid_mask = _map_to_3classes(y)
    X, y5, w = X[valid_mask], y5[valid_mask], w[valid_mask]

    # 1) 先划分 train/test
    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X, y5, w,
        test_size=0.3,
        stratify=y5,
        random_state=RANDOM_STATE
    )

    if 'fat2' in model_name:
        n_est = 3000
        max_depth=15
        gamma=5
        learning_rate=0.01
        reg_lambda=10
        reg_alpha=10
        max_delta_step=10
        min_child_weight=3
    elif 'fat3' in model_name:
        n_est = 3000
        max_depth=6
        gamma=10
        learning_rate=0.01
        reg_lambda=20
        reg_alpha=20
        max_delta_step=5
        min_child_weight=3
    else:
        n_est = 200
        max_depth=10
    
    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        n_estimators=n_est,
        max_depth=max_depth,
        learning_rate=learning_rate,
        gamma=gamma,
        reg_lambda=reg_lambda,
        reg_alpha=reg_alpha,
        min_child_weight=min_child_weight,
        max_delta_step=max_delta_step,
        n_jobs=120,
        random_state=RANDOM_STATE,
        early_stopping_rounds=10
    )

    # 4) 训练（带 sample_weight）
    clf.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        sample_weight_eval_set=[w_train, w_test],
        verbose=True
    )

    # 7) 保存模型
    with open(model_name, "wb") as fout:
        pickle.dump(clf, fout)

    return clf, (X_train, X_test,
                 y_train, y_test,
                 w_train, w_test)

In [ ]:
def _cvm_flatness_value(
    y: np.ndarray,
    Z: np.ndarray,
    w: np.ndarray,
    n_bins: int = 10,
    power: float = 2.0,
) -> float:
    """
    计算 Cramer-von Mises 型 flatness 惩罚的数值（和 _cvm_flatness_neg_grad_wrt_y 对应）：

      L_flat ~ sum_{decor j} sum_{z-bin b} sum_{i in bin b}
                 w_abs_i * |F_b(y_i) - F_global(y_i)|^power

    其中 F_b, F_global 为加权经验 CDF，权重使用 |w|（忽略符号），
    确保 flat_penalty 始终 >= 0。

    参数：
      y      : raw logit，shape (n,)
      Z      : 去相关变量矩阵，shape (n, n_decor)
      w      : 原始样本权重（允许有负权重），shape (n,)
      n_bins : z 的分箱数
      power  : 幂指数，通常取 2

    返回：
      flat_penalty : 一个非负标量，越小表示越“平坦”（越独立）
    """
    y = np.asarray(y, dtype=float).ravel()
    Z = np.asarray(Z, dtype=float)
    w = np.asarray(w, dtype=float).ravel()

    n = y.shape[0]
    if n == 0 or Z.size == 0:
        return 0.0

    if Z.ndim == 1:
        Z = Z.reshape(-1, 1)
    _, n_decor = Z.shape

    # 使用绝对值权重来度量“统计量”，避免负权重导致 flat_penalty < 0
    W_abs = float(np.sum(w) + _EPS)
    if W_abs <= _EPS:
        return 0.0

    # 归一化权重，用于 CDF 计算
    w_norm = w / W_abs

    # 全局经验 CDF 位置 F_global(y)
    global_pos = _weighted_ecdf_positions(y, w_norm)

    flat_penalty = 0.0
    for j in range(n_decor):
        zj = Z[:, j]
        z_min = float(np.min(zj))
        z_max = float(np.max(zj))
        if (not np.isfinite(z_min)) or (not np.isfinite(z_max)) or (z_min == z_max):
            # 这个 decor 变量无有效跨度，跳过
            continue

        edges = np.linspace(z_min, z_max, n_bins + 1, dtype=float)
        bin_idx = np.searchsorted(edges, zj, side="right") - 1
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)

        for b in range(n_bins):
            idx_b = np.nonzero(bin_idx == b)[0]
            # 样本太少的 bin 不做估计，避免统计涨落过大
            if idx_b.size < 3:
                continue

            local_pos = _weighted_ecdf_positions(y[idx_b], w_norm[idx_b])
            diff = local_pos - global_pos[idx_b]

            # power=2 时为平方差；w_norm>=0，保证 flat_penalty >= 0
            flat_penalty += float(np.sum(w_norm[idx_b] * (np.abs(diff) ** power)))

    return float(flat_penalty)



def _weighted_ecdf_positions(y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """
    加权经验分布的位置：对每个样本 i，返回
      pos_i = (累计权重 <= y_i) / sum_j w_j
    实现方式与 hep_ml.losses._compute_positions 相同：
      先按 y 排序，再用归一化权重的累积和构造位置。

    参数：
      y : 一维数组，长度 n
      w : 对应的加权，长度 n（可以是原始权重或归一化权重）

    返回：
      pos : shape (n,) 的位置数组，取值约在 (0, 1) 内
    """
    y = np.asarray(y, dtype=float).ravel()
    w = np.asarray(w, dtype=float).ravel()
    n = y.shape[0]
    if n == 0:
        return np.zeros_like(y)

    # 按 y 排序
    order = np.argsort(y)
    w_sorted = w[order].astype(float)
    # 归一化权重
    W = float(np.sum(w_sorted)) + _EPS
    w_sorted /= W

    # 中点位置：累计权重减去一半本身，避免所有点都挤在边界
    cum = np.cumsum(w_sorted) - 0.5 * w_sorted
    pos_sorted = cum

    # 还原到原始顺序
    pos = np.empty_like(pos_sorted)
    pos[order] = pos_sorted
    return pos


def _cvm_flatness_neg_grad_wrt_y(
    y: np.ndarray,
    groups: Sequence[np.ndarray],
    w: np.ndarray,
    power: float = 2.0
) -> np.ndarray:
    """
    Cramer-von Mises 风格 flatness 惩罚的“负梯度”（近似），
    与 hep_ml 中 BinFlatnessLossFunction 的思想一致：

      L_flat ~ sum_groups sum_{i in group} w_abs_i * |F_group(y_i) - F_global(y_i)|^power

    其中 F_group, F_global 为加权经验 CDF。
    这里对 decor 项使用权重的绝对值 |w|，确保每个事件在 decor 上的贡献为“强度”，
    而不是受 NLO 符号影响。

    参数：
      y      : raw logit，shape (n,)
      groups : 一组 index 数组，每个数组是一段 z-bin 的样本索引
      w      : 原始样本权重（允许有负权重），shape (n,)
      power  : 幂指数，通常取 2；power=2 时是标准的 CvM 型损失

    返回：
      neg_grad : 近似 -dL_flat / dy_i，shape (n,)
    """
    y = np.asarray(y, dtype=float).ravel()
    w = np.asarray(w, dtype=float).ravel()
    n = y.shape[0]
    if n == 0:
        return np.zeros_like(y)
    if groups is None or len(groups) == 0:
        return np.zeros_like(y)

    # 使用绝对值权重来定义 decor 的统计量
    W_abs = float(np.sum(w) + _EPS)
    if W_abs <= _EPS:
        return np.zeros_like(y)

    # 归一化权重用于计算 CDF 位置
    w_norm = w / W_abs

    # 全局位置 F_global(y)
    global_pos = _weighted_ecdf_positions(y, w_norm)

    neg_grad = np.zeros_like(y)
    for idx in groups:
        idx = np.asarray(idx, dtype=int)
        if idx.size < 2:
            continue

        # 本 group 内的局部 CDF 位置 F_group(y_i)
        local_pos = _weighted_ecdf_positions(y[idx], w_norm[idx])
        glob = global_pos[idx]
        diff = local_pos - glob

        # power=2 时 bin_grad = 2 * (local - global)
        bin_grad = power * np.sign(diff) * (np.abs(diff) ** (power - 1.0))
        neg_grad[idx] += bin_grad

    # decor 梯度按 |w| 的归一化权重缩放
    neg_grad *= w_norm
    return neg_grad

def check_weights(w, name="w"):
    w = np.asarray(w, dtype=float).ravel()

    finite_mask = np.isfinite(w)
    if not np.all(finite_mask):
        bad = np.where(~finite_mask)[0]
        print(f"[{name}] 非有限值数量: {bad.size} (nan/inf). 例如索引: {bad[:10].tolist()}")
    else:
        print(f"[{name}] 全部有限 (no nan/inf)")

    n = w.size
    n_pos = int(np.sum(w > 0))
    n_zero = int(np.sum(w == 0))
    n_neg = int(np.sum(w < 0))

    print(f"[{name}] N={n}")
    print(f"[{name}] >0: {n_pos}  ({n_pos/n:.6f})")
    print(f"[{name}] =0: {n_zero} ({n_zero/n:.6f})")
    print(f"[{name}] <0: {n_neg}  ({n_neg/n:.6f})")
    print(f"[{name}] min={np.nanmin(w):.6g}, max={np.nanmax(w):.6g}")
    print(f"[{name}] sum={np.nansum(w):.6g}, sum_abs={np.nansum(np.abs(w)):.6g}")

    if n_neg > 0:
        idx = np.where(w < 0)[0]
        print(f"[{name}] 负权重示例: idx={idx[:10].tolist()}, values={w[idx[:10]]}")


def _resolve_decor_indices(
    X,
    decorrelate_feature_names: Optional[Sequence[Union[str, int]]]
) -> List[int]:
    """
    把用户给的 '需要去相关' 的变量名（或列号）解析为列索引列表。
    支持：
      - 若 X 为 pandas.DataFrame，则直接按列名解析；
      - 否则，若全局定义了 FEATURE_NAMES（list[str]），则从中解析；
      - 否则，若给的是 int（列号），则直接使用；
      - 否则报错。
    """
    if not decorrelate_feature_names:
        return []

    # 1) pandas.DataFrame 情况
    try:
        if isinstance(X, pd.DataFrame):
            name_to_idx = {c: i for i, c in enumerate(list(X.columns))}
            idx = []
            for key in decorrelate_feature_names:
                if isinstance(key, int):
                    idx.append(int(key))
                else:
                    if key not in name_to_idx:
                        raise ValueError(f"去相关变量名 '{key}' 不在 DataFrame 列中。")
                    idx.append(name_to_idx[key])
            return sorted(set(idx))
    except Exception:
        pass

    # 3) 纯 numpy，且用户传了列号
    idx = []
    for key in decorrelate_feature_names:
        if isinstance(key, int):
            idx.append(int(key))
        else:
            raise ValueError(
                "无法从名字解析列索引：X 不是 DataFrame，且未提供全局 FEATURE_NAMES。"
                "你可以：a) 传入 DataFrame；b) 定义全局 FEATURE_NAMES；或 c) 直接传列号(int)。"
            )
    return sorted(set(idx))


def _weighted_stats_centered(y: np.ndarray, z: np.ndarray, w_norm: np.ndarray):
    """
    计算加权均值/中心化/方差/协方差等（基于归一化权重 w_norm, sum=1）
    返回：y0, z0, var_y, var_z, cov, sig_y, sig_z, corr
    """
    y_mean = np.sum(w_norm * y)
    z_mean = np.sum(w_norm * z)
    y0 = y - y_mean
    z0 = z - z_mean
    var_y = np.sum(w_norm * y0 * y0) + _EPS
    var_z = np.sum(w_norm * z0 * z0) + _EPS
    sig_y = np.sqrt(var_y)
    sig_z = np.sqrt(var_z)
    cov = np.sum(w_norm * y0 * z0)
    corr = cov / (sig_y * sig_z + _EPS)
    return y0, z0, var_y, var_z, cov, sig_y, sig_z, corr


def _make_multiclass_objective_with_decor(
    num_class: int,
    Z_train: np.ndarray,      # shape (n_samples, n_decor)
    w_train: np.ndarray,      # shape (n_samples,)
    lam: float,
):
    """
    多类 logloss + CvM flatness 惩罚。

    统一约定：
      - 类内：用原始 w 做加权平均；
      - 类间：各类的“平均损失”直接相加。

    分类项：
      L_cls = sum_k [ sum_{i: y_i=k} w_i * ell_i / sum_{i: y_i=k} w_i ]

    decor 项：
      对每个类 k：
        使用 w_cls_k（只在类 k 非零）计算 CvM L_flat^k，
      总 decor loss = lam * sum_k L_flat^k。

    重要修复：
      1) 由于“类内平均”会让每个 class 的有效权重总和=1，导致 Hessian 总和过小，
         若 min_child_weight>=1 会直接无法生长任何树，训练表现为 loss 完全不变。
         这里对 *整个 objective* 乘一个常数 scale = n_train / num_class，仅改变整体尺度，不改变类间/类内相对权重结构。
      2) XGBoost>=2.1 要求自定义 objective 返回 grad/hess 形状为 (n_samples, n_classes)。
      3) 动态 LR：前 400 轮等效 0.01；第 400-499 轮等效 0.005；之后保持 0.005。
         为保证前 400 轮“完全不变”，用 lr_mult = lr(t)/0.01，只在 grad 上乘该系数。
    """
    Z_train = np.asarray(Z_train, dtype=float)
    w_train = np.asarray(w_train, dtype=float).ravel()
    n_train = w_train.shape[0]

    # === 预先用 Z_train 构造 decor 的分组（每个 decor 变量一个 group 列表） ===
    if Z_train.ndim == 1:
        Z_train = Z_train.reshape(-1, 1)
    n_samples, n_decor = Z_train.shape

    N_DECOR_BINS = 5
    decor_groups_list: List = []
    if lam > 0.0 and n_decor > 0:
        for j in range(n_decor):
            zj = Z_train[:, j]
            z_min = float(np.min(zj))
            z_max = float(np.max(zj))
            if (not np.isfinite(z_min)) or (not np.isfinite(z_max)) or (z_min == z_max):
                decor_groups_list.append(None)
                continue

            edges = np.linspace(z_min, z_max, N_DECOR_BINS + 1, dtype=float)
            bin_idx = np.searchsorted(edges, zj, side="right") - 1
            bin_idx = np.clip(bin_idx, 0, N_DECOR_BINS - 1)

            groups_j: List[np.ndarray] = []
            for b in range(N_DECOR_BINS):
                idx_b = np.nonzero(bin_idx == b)[0]
                if idx_b.size >= 3:
                    groups_j.append(idx_b)

            decor_groups_list.append(groups_j if len(groups_j) > 0 else None)
    else:
        decor_groups_list = [None] * n_decor

    # 关键：整体尺度修复（只影响 scale，不改变你定义的“类内平均/类间相加”的相对结构）
    _scale = float(n_train) / float(num_class) if n_train > 0 else 1.0

    # === 动态 LR 状态（用 objective 被调用次数近似 boosting 轮数；通常每轮训练会调用一次）===
    _base_lr = 0.05
    _iter_state = {"t": 0}

    def obj(y_true: np.ndarray, y_pred_in: np.ndarray):
        """
        sklearn.XGBClassifier 自定义 objective 的签名：
          y_true: shape (n,)
          y_pred_in: raw margin，可能是 (n*num_class,) 或 (n, num_class)

        返回：
          grad, hess: shape (n, num_class)
        """
        # ---- dynamic LR multiplier ----
        t = int(_iter_state["t"])
        _iter_state["t"] = t + 1

        if t < 400:
            lr_t = 0.05
        elif t < 500:
            lr_t = 0.05 - (0.05 - 0.01) * (t - 400) / 100
        else:
            lr_t = 0.01
        lr_mult = float(lr_t) / float(_base_lr)  # 1.0 或 0.5

        y_true = np.asarray(y_true, dtype=int)
        n = y_true.shape[0]
        assert n == n_train, f"训练集大小不一致: n={n}, n_train={n_train}"

        y_pred_in = np.asarray(y_pred_in, dtype=float)
        if y_pred_in.ndim == 1:
            y_pred = y_pred_in.reshape(n, num_class)
        elif y_pred_in.ndim == 2:
            if y_pred_in.shape == (n, num_class):
                y_pred = y_pred_in
            elif y_pred_in.shape == (num_class, n):
                y_pred = y_pred_in.T
            else:
                y_pred = y_pred_in.reshape(n, num_class)
        else:
            y_pred = y_pred_in.reshape(n, num_class)

        # ---------- softmax 概率 ----------
        y_shift = y_pred - np.max(y_pred, axis=1, keepdims=True)
        exp_y = np.exp(y_shift)
        P = exp_y / (np.sum(exp_y, axis=1, keepdims=True) + _EPS)

        y_true_int = y_true
        w_used = w_train  # 与训练集顺序完全对齐

        # ---------- 1) 分类项：类内加权平均，类间直接相加 ----------
        class_w_sum = np.bincount(
            y_true_int,
            weights=w_used,
            minlength=num_class
        ).astype(float)
        class_w_sum[class_w_sum <= _EPS] = _EPS

        # 有效权重：w_eff_i = (w_i / S_{y_i}) * scale
        w_eff = (w_used / class_w_sum[y_true_int]) * _scale

        grad_cls = P.copy()
        grad_cls[np.arange(n), y_true_int] -= 1.0
        grad_cls *= w_eff[:, None]

        hess_cls = (P * (1.0 - P)) * w_eff[:, None]

        # ---------- 2) decor 梯度：每个类的 CvM 梯度，再乘 lam 相加 ----------
        grad_dec = np.zeros_like(grad_cls)
        if lam > 0.0 and n_decor > 0:
            for k in range(num_class):
                mask_k = (y_true_int == k)
                if not np.any(mask_k):
                    continue

                yk = y_pred[:, k]

                w_cls_k = np.zeros_like(w_used)
                w_cls_k[mask_k] = w_used[mask_k]

                gk = np.zeros(n, dtype=float)
                for j in range(n_decor):
                    groups_j = decor_groups_list[j]
                    if groups_j is None:
                        continue
                    neg_grad_flat = _cvm_flatness_neg_grad_wrt_y(
                        yk,
                        groups_j,
                        w_cls_k,
                        power=2.0
                    )
                    gk += (-neg_grad_flat)

                # 与分类项同整体尺度一致
                grad_dec[:, k] = (lam * _scale) * gk

        # ---------- 3) 合并梯度 / Hessian ----------
        grad = grad_cls + grad_dec

        # 动态 LR：只缩放梯度（前 400 轮 lr_mult=1，不改变；之后 lr_mult=0.5）
        grad *= lr_mult

        hess = hess_cls + 1e-6

        grad = np.asarray(grad, dtype=np.float32)
        hess = np.asarray(hess, dtype=np.float32)
        return grad, hess

    return obj



def _make_multiclass_total_loss_metric(
    num_class: int,
    Z_train: np.ndarray,
    Z_test: np.ndarray,
    w_train: np.ndarray,
    w_test: np.ndarray,
    lam: float,
):
    """
    自定义 eval_metric，对应的总损失：

      total_loss = L_cls + lam * sum_k L_flat^{(k)}

    其中：
      1) L_cls：类内加权平均、类间直接相加
      2) decor：对每个类用 w_cls_k 计算 flatness，再类间相加乘 lam

    这里不做 objective 的 scale 修复（metric 保持物理含义与可读性）。
    """
    Z_train = np.asarray(Z_train, dtype=float)
    Z_test  = np.asarray(Z_test,  dtype=float)
    w_train = np.asarray(w_train, dtype=float).ravel()
    w_test  = np.asarray(w_test,  dtype=float).ravel()

    n_train = w_train.shape[0]
    n_test  = w_test.shape[0]

    N_DECOR_BINS = 5

    def feval(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """
        sklearn.XGBClassifier 自定义 eval_metric 签名：
          - y_true: shape (n,)
          - y_pred: shape (n * num_class,) 或 (n, num_class)
                   注意：在某些版本/配置下也可能已经是概率。

        返回一个标量 total_loss，日志里显示为 'feval'。
        """
        y_true = np.asarray(y_true, dtype=int)
        n = y_true.shape[0]

        # 用长度判断是在 train 还是 test 上
        if n == n_train:
            Z = Z_train[:n, :]
            w = w_train[:n]
        elif n == n_test:
            Z = Z_test[:n, :]
            w = w_test[:n]
        else:
            Z = Z_train[:n, :]
            w = w_train[:n]

        y_pred = np.asarray(y_pred, dtype=float)
        if y_pred.ndim == 1:
            y_pred = y_pred.reshape(n, num_class)
        elif y_pred.ndim == 2:
            if y_pred.shape == (n, num_class):
                pass
            elif y_pred.shape == (num_class, n):
                y_pred = y_pred.T
            else:
                y_pred = y_pred.reshape(n, num_class)
        else:
            y_pred = y_pred.reshape(n, num_class)

        # --- 判断 y_pred 是否已经是概率（稳妥兼容） ---
        # 若每行和≈1 且都在 [0,1]，则认为是 prob；否则认为是 raw margin 需要 softmax
        row_sum = np.sum(y_pred, axis=1)
        is_prob = (
            np.all(np.isfinite(y_pred)) and
            np.all(y_pred >= -1e-6) and np.all(y_pred <= 1.0 + 1e-6) and
            np.all(np.isfinite(row_sum)) and
            np.mean(np.abs(row_sum - 1.0)) < 1e-3
        )

        if is_prob:
            P = np.clip(y_pred, _EPS, 1.0)
            P = P / (np.sum(P, axis=1, keepdims=True) + _EPS)
        else:
            y_shift = y_pred - np.max(y_pred, axis=1, keepdims=True)
            exp_y = np.exp(y_shift)
            P = exp_y / (np.sum(exp_y, axis=1, keepdims=True) + _EPS)

        y_true_int = y_true
        w = np.asarray(w, dtype=float).ravel()

        # ---------- 1) 分类 logloss：类内加权平均，类间直接相加 ----------
        class_w_sum = np.bincount(
            y_true_int,
            weights=w,
            minlength=num_class
        ).astype(float)
        class_w_sum[class_w_sum <= _EPS] = _EPS

        w_eff = w / class_w_sum[y_true_int]

        p_true = P[np.arange(n), y_true_int]
        ell = -np.log(p_true + _EPS)
        logloss = float(np.sum(w_eff * ell))

        # ---------- 2) decor：lam * sum_k L_flat^{(k)} ----------
        flat_penalty = 0.0
        if lam > 0.0 and Z.size > 0:
            for k in range(num_class):
                mask_k = (y_true_int == k)
                if not np.any(mask_k):
                    continue

                w_cls_k = np.zeros_like(w)
                w_cls_k[mask_k] = w[mask_k]

                yk = y_pred[:, k] if not is_prob else np.log(P[:, k] + _EPS)  # prob 情况下用 logit-like 也更合理
                L_flat_k = _cvm_flatness_value(
                    yk,
                    Z,
                    w_cls_k,
                    n_bins=N_DECOR_BINS,
                    power=2.0,
                )
                flat_penalty += L_flat_k

            flat_penalty *= lam

        total_loss = logloss + flat_penalty
        return float(total_loss)

    return feval

In [ ]:
def train_multi_model_un(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    model_name: str,
    decorrelate_feature_names: Optional[Sequence[Union[str, int]]] = None,  # ← 指定需“无关”的变量（用名字，或索引）
):
    """
    多分类（BKG/VVV/VH/...）训练：在标准多类 logloss 的基础上，
    对指定的 'decorrelate_feature_names' 变量与各类别 raw logit 的 Pearson 相关性加入惩罚项，
    并且这些变量不作为训练输入特征（仅用于惩罚项）。

    返回：
      clf, (X_train_used, X_test_used, y_train, y_test, w_train, w_test)
    """
    X = np.asarray(X)
    y = np.asarray(y)
    w = np.asarray(w, dtype=float)

    # 0) 将原始标签映射为 3→5 类等（与你现有实现一致）
    y5, valid_mask = _map_to_3classes(y)  # 保持调用与返回不变
    X, y5, w = X[valid_mask], y5[valid_mask], w[valid_mask]

    # 1) 先划分 train/test（在“剔除列”之前做，以便 Z_train/Z_test 与训练/验证严格同序）
    X_train_all, X_test_all, y_train, y_test, w_train, w_test = train_test_split(
        X, y5, w,
        test_size=0.3,
        stratify=y5,
        random_state=RANDOM_STATE
    )

    # 2) 解析去相关列索引，并构造“训练输入特征”与“仅用于惩罚的 Z”
    decor_idx = _resolve_decor_indices(X_train_all, decorrelate_feature_names)
    if len(decor_idx) > 0:
        all_idx = np.arange(X_train_all.shape[1])
        keep_idx = np.setdiff1d(all_idx, np.array(decor_idx, dtype=int), assume_unique=False)
        if keep_idx.size == 0:
            raise ValueError("去相关列覆盖了所有特征：没有可用于训练的输入。请减少去相关变量数量。")

        # 训练输入（剔除了去相关变量）
        X_train = X_train_all[:, keep_idx]
        X_test  = X_test_all[:, keep_idx]
        # 仅用于惩罚的 Z
        Z_train = X_train_all[:, decor_idx]
        Z_test  = X_test_all[:, decor_idx]
    else:
        # 未指定去相关变量：退化为原始训练
        X_train, X_test = X_train_all, X_test_all
        Z_train = np.zeros((X_train.shape[0], 0), dtype=float)  # shape (n, 0)
        Z_test  = np.zeros((X_test.shape[0], 0), dtype=float)

    # 在真正训练前，打印模型实际看到的输入维度
    print(f"[train_multi_model_un] X_train shape used as model input: {X_train.shape}")
    print(f"[train_multi_model_un] #features used by model: {X_train.shape[1]}")
    print(f"[train_multi_model_un] Z_train shape (decor vars for penalty): {Z_train.shape}")

    
    # 3) 模型超参（保持原有逻辑）
    n_est = 200
    max_depth = 10
    gamma = 0.0
    learning_rate = 0.1
    reg_lambda = 1.0
    reg_alpha = 0.0
    #max_delta_step = 0.0
    min_child_weight = 1.0

    if 'fat2' in model_name:
        n_est = 1500
        max_depth = 10
        gamma = 5
        learning_rate = 0.01
        reg_lambda = 10
        reg_alpha = 10
        #max_delta_step = 20
        min_child_weight = 5
    elif 'fat3' in model_name:
        n_est = 1200
        max_depth = 8
        gamma = 5
        learning_rate = 0.01
        reg_lambda = 10
        reg_alpha = 10
        #max_delta_step = 10
        min_child_weight = 3
    else:
        # (保持上面的默认值即可)
        pass

    # === 通用 XGBoost 参数（不改变你现有的训练超参） ===
    # n_jobs 用一个相对合理的线程数，避免 2048 造成调度反而变慢
    max_threads = os.cpu_count() or 1
    n_threads = max(1, min(16, max_threads))
    print(f"n_threads: {n_threads}")

    
    # 4) 构建分类器
    num_classes = 5  # 与原实现一致
    custom_obj = None

    common_xgb_kwargs = dict(
        num_class=num_classes,
        n_estimators=n_est,
        max_depth=max_depth,
        learning_rate=learning_rate,
        gamma=gamma,
        reg_lambda=reg_lambda,
        reg_alpha=reg_alpha,
        min_child_weight=min_child_weight,
        #max_delta_step=max_delta_step,
        n_jobs=n_threads,
        random_state=RANDOM_STATE,
    )

    # 优先尝试 GPU（需要编译了 GPU 支持的 xgboost），失败则回退到 CPU hist
    gpu_kwargs = dict(
        tree_method="hist",
        #predictor="gpu_predictor",
        device = "cuda"
    )
    cpu_kwargs = dict(
        tree_method="hist",   # CPU 上使用高效直方图算法
    )

    if Z_train.shape[1] > 0 and DECOR_LAMBDA > 0.0:
        # 自定义目标：多类 logloss + decor penalty + 动态学习率
        custom_obj = _make_multiclass_objective_with_decor(
            num_class=num_classes,
            Z_train=Z_train,
            w_train=w_train,
            lam=DECOR_LAMBDA,
            #base_learning_rate=learning_rate,
        )

        # 自定义 eval_metric：总损失（logloss + decor）
        total_loss_metric = _make_multiclass_total_loss_metric(
            num_class=num_classes,
            Z_train=Z_train,
            Z_test=Z_test,
            w_train=w_train,
            w_test=w_test,
            lam=DECOR_LAMBDA,
        )

        # === 优先尝试 GPU 版本，失败则自动回退到 CPU-hist ===
        try:
            clf = XGBClassifier(
                objective=custom_obj,
                eval_metric=total_loss_metric,
                early_stopping_rounds=10,
                **common_xgb_kwargs,
                **gpu_kwargs,
            )
        except xgb.core.XGBoostError:
            clf = XGBClassifier(
                objective=custom_obj,
                eval_metric=total_loss_metric,
                early_stopping_rounds=10,
                **common_xgb_kwargs,
                **cpu_kwargs,
            )

    else:
        # 未去相关时，保持原本的多类 softprob
        try:
            clf = XGBClassifier(
                objective="multi:softprob",
                early_stopping_rounds=10,
                **common_xgb_kwargs,
                **gpu_kwargs,
            )
        except xgb.core.XGBoostError:
            clf = XGBClassifier(
                objective="multi:softprob",
                early_stopping_rounds=10,
                **common_xgb_kwargs,
                **cpu_kwargs,
            )

    # 5) 训练（decor 分支权重在 objective/metric 内部处理）
    if Z_train.shape[1] > 0 and DECOR_LAMBDA > 0.0 and custom_obj is not None:
        clf.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_test, y_test)],
            verbose=True
        )
    else:
        clf.fit(
            X_train, y_train,
            sample_weight=w_train,
            eval_set=[(X_train, y_train), (X_test, y_test)],
            sample_weight_eval_set=[w_train, w_test],
            verbose=True
        )

    # 6) 保存模型
    if Z_train.shape[1] > 0 and DECOR_LAMBDA > 0.0 and custom_obj is not None:
        clf.save_model(model_name if model_name.endswith(".json") else model_name + ".json")
    else:
        with open(model_name, "wb") as fout:
            pickle.dump(clf, fout)

    # 返回的 X_train/X_test 即为“实际用于训练的特征”（已剔除去相关列）
    return clf, (X_train_all, X_test_all, y_train, y_test, w_train, w_test)

In [ ]:
def plot_multi_results(clf, splits, tree_name):
    """
    适配 5-class BDT 的可视化：
      (1) ROC：VVV 与其余 4 类分别的 ROC；VH 与其余 4 类分别的 ROC。
          同一目标类的 4 条曲线画到一张图里；Test=实线，Train=虚线；手动四种颜色。
      (2) 特征重要性：保持原逻辑与风格（x 轴 log）。
      (3) Score 分布：VVV vs rest、VH vs rest（保持 one-vs-rest）。
      (4) Loss 曲线、特征相关矩阵：保持原逻辑。
    """
    X_train, X_test, y_train, y_test, w_train, w_test = splits

    IDX_VVV, IDX_VH, IDX_VV, IDX_TT, IDX_QCD = 0, 1, 2, 3#, 4
    class_names = ["VVV", "TT", "VV", "QCD"] #"VH", 
    if tree_name == 'fat2':
        feature_names = list(branches2)
    elif tree_name == 'fat3':
        feature_names = list(branches3)
    else:
        feature_names = [f"f{i}" for i in range(clf.n_features_in_)]

    # ---------- 1. ROC：对目标类与每个对手类分别计算 one-vs-one ----------
    # 评分采用 p_target / (p_target + p_opponent)，并仅在 (target, opponent) 两类样本上评估
    def _roc_one_vs_one(Xs, ys, ws, target_idx, opp_idx):
        probs = clf.predict_proba(Xs)  # shape: (n, 5)
        mask = (ys == target_idx) | (ys == opp_idx)
        if not np.any(mask):
            return None
        y_bin = (ys[mask] == target_idx).astype(int)
        if (y_bin.sum() == 0) or (y_bin.sum() == len(y_bin)):
            return None
        p_t = probs[mask, target_idx]
        p_o = probs[mask, opp_idx]
        eps = 1e-12
        score = p_t / np.clip(p_t + p_o, eps, None)
        auc = roc_auc_score(y_bin, score, sample_weight=ws[mask])
        fpr, tpr, _ = roc_curve(y_bin, score, sample_weight=ws[mask])
        return fpr, tpr, auc

    # 手动 4 种颜色（与 4 个对手一一对应）
    palette = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple']

    def _plot_all_opponents_for_target(target_idx, target_name):
        # 收集对手索引（除自身外的 4 类）
        opponents = [i for i in range(5) if i != target_idx]
        plt.figure(figsize=(10, 10))

        # Test=实线，Train=虚线；每个对手一组颜色
        for color, opp_idx in zip(palette, opponents):
            opp_name = class_names[opp_idx]

            r_test = _roc_one_vs_one(X_test, y_test, w_test, target_idx, opp_idx)
            r_train = _roc_one_vs_one(X_train, y_train, w_train, target_idx, opp_idx)

            if r_test is not None:
                fpr_tst, tpr_tst, auc_tst = r_test
                plt.plot(tpr_tst, fpr_tst, color=color, linestyle='-',
                         label=f"{opp_name} (Test AUC={auc_tst:.3f})")
                print(f"{tree_name} Test AUC ({target_name} vs {opp_name}) = {auc_tst:.4f}")
            else:
                print(f"{tree_name} Test AUC ({target_name} vs {opp_name}) = N/A")

            if r_train is not None:
                fpr_tr, tpr_tr, auc_tr = r_train
                plt.plot(tpr_tr, fpr_tr, color=color, linestyle='--',
                         label=f"{opp_name} (Train AUC={auc_tr:.3f})")
                print(f"{tree_name} Train AUC ({target_name} vs {opp_name}) = {auc_tr:.4f}")
            else:
                print(f"{tree_name} Train AUC ({target_name} vs {opp_name}) = N/A")

        plt.xlabel(rf"$\epsilon_{{\rm {target_name}}}$", fontsize=20)
        plt.ylabel(r"$\epsilon_{\rm bkg}$", fontsize=20)
        plt.yscale("log")
        plt.ylim(1e-6, 1)
        plt.xlim(0, 1)
        ####plt.title(f"{tree_name} ROC ({target_name} vs each opponent)", fontsize=16)
        plt.legend(loc="lower right", fontsize=22)
        plt.tight_layout()
        plt.show()

    # 画两张：VVV vs 4 类、VH vs 4 类
    _plot_all_opponents_for_target(IDX_VVV, "VVV")
    #_plot_all_opponents_for_target(IDX_VH,  "VH")

    # ---------- 2. 特征重要性（保持不变，x 轴 log） ----------
    fig, ax = plt.subplots(figsize=(10, 20))
    plot_importance(clf, ax=ax, max_num_features=200)
    ax.set_title(f"{tree_name} Feature Importance", fontsize=16)
    ax.set_xscale("log")
    widths = [patch.get_width() for patch in ax.patches]
    if len([w for w in widths if w > 0]) > 0:
        min_nonzero = min(w for w in widths if w > 0)
        max_width = max(widths) if widths else 1.0
        ax.set_xlim(min_nonzero / 2, max_width * 2)
    try:
        raw_labels = [t.get_text() for t in ax.get_yticklabels()]
        mapped = []
        for s in raw_labels:
            if s.startswith('f') and s[1:].isdigit():
                i = int(s[1:])
                mapped.append(feature_names[i] if i < len(feature_names) else s)
            else:
                mapped.append(s)
        if mapped:
            ax.set_yticklabels(mapped)
    except Exception:
        pass
    plt.tight_layout()
    plt.show()

    # ---------- 3. Score 分布：VVV vs rest、VH vs rest ----------
    probs_train = clf.predict_proba(X_train)
    probs_test  = clf.predict_proba(X_test)

    def _plot_score_dist_for_target(target_idx, target_name, bkg_name):
        p_train = probs_train[:, target_idx]
        p_test  = probs_test[:, target_idx]

        train_pos   = p_train[y_train == target_idx]
        train_w_pos = w_train[y_train == target_idx]
        train_bkg   = p_train[y_train != target_idx]
        train_w_bkg = w_train[y_train != target_idx]

        test_pos   = p_test[y_test == target_idx]
        test_w_pos = w_test[y_test == target_idx]
        test_bkg   = p_test[y_test != target_idx]
        test_w_bkg = w_test[y_test != target_idx]

        plt.figure()
        plt.xlim(0, 1)
        bin_edges = np.linspace(0, 1, 31)
        plt.hist(train_bkg, bins=bin_edges, weights=train_w_bkg,
                 density=True, histtype="bar", alpha=0.5, label=f"Train {bkg_name}")
        plt.hist(train_pos, bins=bin_edges, weights=train_w_pos,
                 density=True, histtype="bar", alpha=0.5, label=f"Train {target_name}")
        plt.hist(test_bkg, bins=bin_edges, weights=test_w_bkg,
                 density=True, histtype="step", linewidth=2, alpha=0.7,
                 label=f"Test {bkg_name}", color="lime")
        plt.hist(test_pos, bins=bin_edges, weights=test_w_pos,
                 density=True, histtype="step", linewidth=2, alpha=0.7,
                 label=f"Test {target_name}", color="red")
        plt.xlabel("BDT Score")
        plt.yscale("log")
        plt.ylim(1e-2,)
        plt.ylabel("Density")
        ###plt.title(f"{tree_name} Score Distribution ({target_name} vs {bkg_name})", fontsize=16)
        plt.legend()
        plt.show()

    _plot_score_dist_for_target(IDX_VVV, "VVV", "VH+VV+TT+QCD")
    _plot_score_dist_for_target(IDX_VH,  "VH",  "VVV+VV+TT+QCD")

    # ---------- 4. Loss 曲线（兼容 multi-class 的 mlogloss） ----------
    try:
        evals = clf.evals_result() if hasattr(clf, 'evals_result') else clf.evals_result_
    except Exception:
        evals = None

    if evals is not None and 'validation_0' in evals and 'validation_1' in evals:
        keys0 = list(evals['validation_0'].keys())
        loss_key = None
        for k in keys0:
            if 'logloss' in k:  # 'logloss' 或 'mlogloss'
                loss_key = k
                break
        if loss_key is not None:
            train_loss = evals['validation_0'][loss_key]
            test_loss  = evals['validation_1'][loss_key]
            epochs = range(1, len(train_loss) + 1)

            plt.figure(figsize=(8, 5))
            plt.plot(epochs, train_loss, label='Train')
            plt.plot(epochs, test_loss,  label='Test')
            plt.xlabel('Epoch')
            plt.ylabel(loss_key)
            ###plt.title(f'{tree_name} Loss vs. Epoch')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.5)
            plt.show()

    # ---------- 5. 特征相关性矩阵（保持原逻辑） ----------
    X_train_df = pd.DataFrame(X_train, columns=feature_names)
    corr = X_train_df.corr(numeric_only=True)
    corr = corr.dropna(axis=0, how='all').dropna(axis=1, how='all')
    plt.figure(figsize=(20, 20))
    plt.imshow(corr.values, aspect='equal', interpolation='none')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=10)
    plt.yticks(range(len(corr.index)),    corr.index, fontsize=10)
    ###plt.title(f"{tree_name} Feature Correlation Matrix", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_multi_results_un(clf, splits, tree_name, decorrelate_feature_names=None):
    """
    适配 5-class BDT 的可视化：
      (1) ROC：VVV 与其余 4 类分别的 ROC；VH 与其余 4 类分别的 ROC。
      (2) 特征重要性：用真实特征名标注（与训练输入一致），x 轴 log。
      (3) Score 分布：VVV vs rest、VH vs rest（one-vs-rest）。
      (4) Loss 曲线、特征相关矩阵（基于训练输入）。
      (5) 去相关检验（仅此处使用“包含 decor 特征”的 X_full）：
          5.1 【加权】相关系数热力图：5 个 class 的 score vs decor 变量（Train/Test）
          5.2 【加权】scatter：5 个 class 的 score vs decor 变量（Test）
          5.3 【加权】decor 变量分布：对每个 true class，用该 class 自己的 score 做阈值选择，比较 decor 分布（Test）
    """

    X_train_full, X_test_full, y_train, y_test, w_train, w_test = splits

    # ====== 0) 取得“全特征名”与 decor 索引；构造“训练使用的特征名/矩阵” ======
    def _get_full_feature_names(X_like):
        if hasattr(X_like, "columns"):
            return list(X_like.columns)
        if tree_name == "fat2":
            return list(branches2)
        elif tree_name == "fat3":
            return list(branches3)
        ncols = X_like.shape[1]
        return [f"f{i}" for i in range(ncols)]

    full_feature_names = _get_full_feature_names(X_train_full)

    def _resolve_indices(names_or_idx, feature_names_list):
        if not names_or_idx:
            return []
        name_to_idx = {c: i for i, c in enumerate(feature_names_list)}
        out = []
        for key in names_or_idx:
            if isinstance(key, int):
                if 0 <= key < len(feature_names_list):
                    out.append(key)
                else:
                    print(f"[WARN] decor 列号 {key} 越界，已跳过。")
            else:
                if key in name_to_idx:
                    out.append(name_to_idx[key])
                else:
                    print(f"[INFO] decor 变量 '{key}' 不在全特征列表中，跳过。")
        seen, res = set(), []
        for i in out:
            if i not in seen:
                seen.add(i)
                res.append(i)
        return res

    decor_idx_full = _resolve_indices(decorrelate_feature_names, full_feature_names)
    all_idx = np.arange(len(full_feature_names))
    keep_idx = np.setdiff1d(all_idx, np.array(decor_idx_full, dtype=int))  # 训练真正使用的列

    def _slice_cols(X_like, idx):
        if hasattr(X_like, "iloc"):
            return X_like.iloc[:, idx]
        return X_like[:, idx]

    X_train_used = _slice_cols(X_train_full, keep_idx)
    X_test_used  = _slice_cols(X_test_full,  keep_idx)

    feat_names_used = [full_feature_names[i] for i in keep_idx]

    if hasattr(clf, "n_features_in_") and getattr(clf, "n_features_in_") != X_train_used.shape[1]:
        if hasattr(X_train_full, "shape") and X_train_full.shape[1] == getattr(clf, "n_features_in_"):
            X_train_used, X_test_used = X_train_full, X_test_full
            feat_names_used = full_feature_names
            decor_idx_full = _resolve_indices([], full_feature_names)  # decor 检验将被跳过
            print("[INFO] 检测到 splits 已与模型输入对齐，按原 X 使用。")
        else:
            raise ValueError(
                f"特征数与模型不一致：X_used={X_train_used.shape[1]} vs clf.n_features_in_={getattr(clf,'n_features_in_',None)}"
            )

    # ====== 1) 常量与工具 ======
    IDX_VVV, IDX_VH, IDX_TT, IDX_VV, IDX_QCD = 0, 1, 2, 3, 4
    class_names = ["VVV", "VH", "TT", "VV", "QCD"]
    n_classes = len(class_names)

    def _as_array(Xlike):
        return Xlike.to_numpy() if hasattr(Xlike, "to_numpy") else np.asarray(Xlike)

    def _safe_w(w):
        w = np.asarray(w, float).ravel()
        w = np.abs(w)
        return w

    def _weighted_pearson(x, y, w, eps=1e-12):
        x = np.asarray(x, float).ravel()
        y = np.asarray(y, float).ravel()
        w = _safe_w(w)

        m = np.isfinite(x) & np.isfinite(y) & np.isfinite(w)
        if not np.any(m):
            return 0.0
        x = x[m]
        y = y[m]
        w = w[m]

        sw = w.sum()
        if sw <= eps:
            return 0.0

        wx = (w * x).sum() / (sw + eps)
        wy = (w * y).sum() / (sw + eps)
        x0 = x - wx
        y0 = y - wy
        cov = (w * x0 * y0).sum() / (sw + eps)
        vx = (w * x0 * x0).sum() / (sw + eps)
        vy = (w * y0 * y0).sum() / (sw + eps)
        return float(cov / (np.sqrt(vx * vy) + eps))

    def _weighted_quantile(x, q, w, eps=1e-12):
        x = np.asarray(x, float).ravel()
        w = _safe_w(w)
        q = np.asarray(q, float)

        m = np.isfinite(x) & np.isfinite(w)
        if not np.any(m):
            return np.full_like(q, np.nan, dtype=float)

        x = x[m]
        w = w[m]
        sw = w.sum()
        if sw <= eps:
            return np.quantile(x, q)

        order = np.argsort(x)
        x_sorted = x[order]
        w_sorted = w[order]
        cw = np.cumsum(w_sorted)
        cw = cw / (cw[-1] + eps)
        return np.interp(q, cw, x_sorted)

    # ====== 1.5) 【关键修正】统一预测输出为“概率”，并对齐 early-stopping 的 best_iteration ======
    def _get_xgb_booster(model):
        # xgboost.Booster
        if hasattr(xgb, "Booster") and isinstance(model, xgb.Booster):
            return model

        # 你文件里自带的 _BoosterWrapper：有 _booster 属性
        if hasattr(model, "_booster") and isinstance(getattr(model, "_booster"), xgb.Booster):
            return getattr(model, "_booster")

        # sklearn wrapper: XGBClassifier / XGBRegressor
        if hasattr(model, "get_booster"):
            try:
                return model.get_booster()
            except Exception:
                return None
        return None

    def _try_enable_gpu_inference(model, device="cuda"):
        """
        尝试把推理切到 GPU（失败自动回退 CPU）。
        注意：XGBoost 2.0+ 推荐用 device='cuda'。
        """
        booster = _get_xgb_booster(model)
        if booster is None:
            return False
        try:
            booster.set_param({"device": device})
            if hasattr(model, "set_params"):
                try:
                    model.set_params(device=device)
                except Exception:
                    pass
            return True
        except Exception as e:
            print(f"[WARN] 无法启用 GPU 推理（{e}），将回退到 CPU。")
            return False

    def _sigmoid(x):
        x = np.asarray(x, float)
        return 1.0 / (1.0 + np.exp(-x))

    def _softmax(m):
        m = np.asarray(m, float)
        m = m - np.max(m, axis=1, keepdims=True)
        e = np.exp(m)
        return e / np.sum(e, axis=1, keepdims=True)

    def _get_best_iteration(model, booster):
        """
        尽量拿到 early-stopping 的 best_iteration（若不存在则返回 None）。
        """
        for obj in (model, booster):
            bi = getattr(obj, "best_iteration", None)
            if bi is not None:
                try:
                    bi = int(bi)
                    if bi >= 0:
                        return bi
                except Exception:
                    pass

        # 有些版本把 best_iteration 存在 booster.attributes() 里
        try:
            attrs = booster.attributes()
            if isinstance(attrs, dict) and "best_iteration" in attrs:
                bi = int(attrs["best_iteration"])
                if bi >= 0:
                    return bi
        except Exception:
            pass
        return None

    def _ensure_proba(pred, n_classes_local):
        """
        把 pred 统一成概率：
          - 若看起来像 raw margin/logit：做 softmax（多分类）/ sigmoid（二分类）
          - 若已经是概率：原样返回
        """
        pred = np.asarray(pred)

        # 可能出现 (n*nclass,) 的 1D 扁平输出：先 reshape
        if pred.ndim == 1 and n_classes_local > 1 and (pred.size % n_classes_local == 0):
            pred = pred.reshape(-1, n_classes_local)

        if pred.ndim == 1:
            # binary 情况
            if pred.min() < 0.0 or pred.max() > 1.0:
                p1 = _sigmoid(pred)
            else:
                p1 = pred
            return np.vstack([1.0 - p1, p1]).T

        # multiclass：判断是不是概率
        row_sum = pred.sum(axis=1)
        if (pred.min() < 0.0) or (pred.max() > 1.0) or (not np.allclose(row_sum, 1.0, rtol=1e-3, atol=1e-3)):
            pred = _softmax(pred)

        return pred

    def _predict_proba_fast(model, Xlike):
        """
        统一预测概率入口：
          - 优先走 Booster.inplace_predict（更快；device=cuda 时可走 GPU）
          - 然后把输出统一成“概率”（非常关键：避免 raw margin 导致 ROC/AUC/score 分布不一致）
          - 若存在 best_iteration：使用 iteration_range 对齐 early-stopping 的 best 模型
        """
        Xarr = _as_array(Xlike)

        booster = _get_xgb_booster(model)
        if booster is None:
            # 兜底：走模型自带 predict_proba（与你 train.py 的行为一致）
            return model.predict_proba(Xlike)

        best_it = _get_best_iteration(model, booster)
        iter_range = (0, best_it + 1) if (best_it is not None) else None

        # 优先 inplace_predict
        if hasattr(booster, "inplace_predict"):
            try:
                kwargs = dict(
                    predict_type="value",
                    strict_shape=True,
                    validate_features=False,
                )
                if iter_range is not None:
                    kwargs["iteration_range"] = iter_range
                pred = booster.inplace_predict(Xarr, **kwargs)
            except TypeError:
                # 老版本不支持 iteration_range/validate_features 等参数
                pred = booster.inplace_predict(Xarr)
        else:
            # 极老版本兜底
            try:
                dmat = xgb.DMatrix(Xarr)
                if iter_range is not None:
                    pred = booster.predict(dmat, iteration_range=iter_range)
                else:
                    pred = booster.predict(dmat)
            except Exception:
                return model.predict_proba(Xlike)

        pred = _ensure_proba(pred, n_classes)
        return pred

    gpu_ok = _try_enable_gpu_inference(clf, device="cuda")
    if gpu_ok:
        print("[INFO] 已请求 XGBoost 使用 GPU 进行推理（device=cuda）。")

    # 只做一次预测，后面全部复用
    probs_train = _predict_proba_fast(clf, X_train_used)  # shape: (n_train, 5)
    probs_test  = _predict_proba_fast(clf, X_test_used)   # shape: (n_test, 5)

    # 可选：做个一致性 sanity check（不影响流程）
    if probs_train.ndim != 2 or probs_train.shape[1] != n_classes:
        raise ValueError(f"predict_proba 输出维度异常：{probs_train.shape}，期望 (N, {n_classes})")
    if not np.allclose(probs_test.sum(axis=1), 1.0, rtol=1e-3, atol=1e-3):
        print("[WARN] probs_test 行和不接近 1：说明仍可能不是概率输出（请检查 xgboost 版本/模型类型）。")
    
    # ====== 2) ROC（复用 probs_*，不再在每个 opponent 里重复预测）======
    def _roc_one_vs_one_from_probs(probs, ys, ws, target_idx, opp_idx):
        mask = (ys == target_idx) | (ys == opp_idx)
        if not np.any(mask):
            return None
        y_bin = (ys[mask] == target_idx).astype(int)
        if (y_bin.sum() == 0) or (y_bin.sum() == len(y_bin)):
            return None

        p_t = probs[mask, target_idx]
        p_o = probs[mask, opp_idx]
        eps = 1e-12
        score = p_t / np.clip(p_t + p_o, eps, None)

        auc = roc_auc_score(y_bin, score, sample_weight=ws[mask])
        fpr, tpr, _ = roc_curve(y_bin, score, sample_weight=ws[mask])
        return fpr, tpr, auc

    palette = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple']

    def _plot_all_opponents_for_target(target_idx, target_name):
        opponents = [i for i in range(n_classes) if i != target_idx]
        plt.figure(figsize=(10, 10))
        for color, opp_idx in zip(palette, opponents):
            opp_name = class_names[opp_idx]

            r_test = _roc_one_vs_one_from_probs(probs_test,  y_test,  w_test,  target_idx, opp_idx)
            r_train = _roc_one_vs_one_from_probs(probs_train, y_train, w_train, target_idx, opp_idx)

            if r_test is not None:
                fpr_tst, tpr_tst, auc_tst = r_test
                plt.plot(tpr_tst, fpr_tst, color=color, linestyle='-',
                         label=f"{opp_name} (Test AUC={auc_tst:.3f})")
                print(f"{tree_name} Test AUC ({target_name} vs {opp_name}) = {auc_tst:.4f}")
            else:
                print(f"{tree_name} Test AUC ({target_name} vs {opp_name}) = N/A")

            if r_train is not None:
                fpr_tr, tpr_tr, auc_tr = r_train
                plt.plot(tpr_tr, fpr_tr, color=color, linestyle='--',
                         label=f"{opp_name} (Train AUC={auc_tr:.3f})")
                print(f"{tree_name} Train AUC ({target_name} vs {opp_name}) = {auc_tr:.4f}")
            else:
                print(f"{tree_name} Train AUC ({target_name} vs {opp_name}) = N/A")

        plt.xlabel(rf"$\epsilon_{{\rm {target_name}}}$", fontsize=20)
        plt.ylabel(r"$\epsilon_{\rm bkg}$", fontsize=20)
        plt.yscale("log")
        plt.ylim(1e-6, 1)
        plt.xlim(0, 1)
        plt.legend(loc="lower right", fontsize=22)
        plt.tight_layout()
        plt.show()

    _plot_all_opponents_for_target(IDX_VVV, "VVV")
    # _plot_all_opponents_for_target(IDX_VH, "VH")

    # ====== 3) 特征重要性（用真实特征名，且与训练输入一致）======
    fig, ax = plt.subplots(figsize=(10, 20))
    plot_importance(clf, ax=ax, max_num_features=200)
    ax.set_title(f"{tree_name} Feature Importance", fontsize=16)
    ax.set_xscale("log")
    widths = [patch.get_width() for patch in ax.patches]
    if len([w for w in widths if w > 0]) > 0:
        min_nonzero = min(w for w in widths if w > 0)
        max_width = max(widths) if widths else 1.0
        ax.set_xlim(min_nonzero / 2, max_width * 2)

    try:
        raw_labels = [t.get_text() for t in ax.get_yticklabels()]
        mapped = []
        for s in raw_labels:
            if isinstance(s, str) and s.startswith('f'):
                num = s[1:]
                if num.isdigit():
                    i = int(num)
                    mapped.append(feat_names_used[i] if i < len(feat_names_used) else s)
                else:
                    mapped.append(s)
            else:
                mapped.append(s)
        if mapped:
            ax.set_yticklabels(mapped)
    except Exception:
        pass
    plt.tight_layout()
    plt.show()

    # ====== 4) Score 分布（基于 probs_* 缓存）======
    def _plot_score_dist_for_target(target_idx, target_name, bkg_name):
        p_train = probs_train[:, target_idx]
        p_test  = probs_test[:, target_idx]

        train_pos   = p_train[y_train == target_idx]
        train_w_pos = w_train[y_train == target_idx]
        train_bkg   = p_train[y_train != target_idx]
        train_w_bkg = w_train[y_train != target_idx]

        test_pos   = p_test[y_test == target_idx]
        test_w_pos = w_test[y_test == target_idx]
        test_bkg   = p_test[y_test != target_idx]
        test_w_bkg = w_test[y_test != target_idx]

        plt.figure()
        plt.xlim(0, 1)
        bin_edges = np.linspace(0, 1, 31)
        plt.hist(train_bkg, bins=bin_edges, weights=train_w_bkg,
                 density=True, histtype="bar", alpha=0.5, label=f"Train {bkg_name}")
        plt.hist(train_pos, bins=bin_edges, weights=train_w_pos,
                 density=True, histtype="bar", alpha=0.5, label=f"Train {target_name}")
        plt.hist(test_bkg, bins=bin_edges, weights=test_w_bkg,
                 density=True, histtype="step", linewidth=2, alpha=0.7,
                 label=f"Test {bkg_name}", color="lime")
        plt.hist(test_pos, bins=bin_edges, weights=test_w_pos,
                 density=True, histtype="step", linewidth=2, alpha=0.7,
                 label=f"Test {target_name}", color="red")
        plt.xlabel("BDT Score")
        plt.yscale("log")
        plt.ylim(1e-2,)
        plt.ylabel("Density")
        #plt.title(f"{tree_name} Score Distribution ({target_name} vs {bkg_name})", fontsize=16)
        plt.legend()
        plt.show()

    _plot_score_dist_for_target(IDX_VVV, "VVV", "VH+VV+TT+QCD")
    _plot_score_dist_for_target(IDX_VH,  "VH",  "VVV+VV+TT+QCD")

    # ====== 5) Loss 曲线（总 loss）======
    try:
        evals = clf.evals_result() if hasattr(clf, 'evals_result') else clf.evals_result_
    except Exception:
        evals = None

    if evals is not None and 'validation_0' in evals and 'validation_1' in evals:
        keys0 = list(evals['validation_0'].keys())
        loss_key = None
        for k in keys0:
            if 'total_loss' in k:
                loss_key = k
                break
        if loss_key is None:
            for k in keys0:
                if 'feval' in k:
                    loss_key = k
                    break
        if loss_key is None:
            for k in keys0:
                if 'logloss' in k:
                    loss_key = k
                    break

        if loss_key is not None:
            train_loss = evals['validation_0'][loss_key]
            test_loss  = evals['validation_1'][loss_key]
            epochs = range(1, len(train_loss) + 1)

            plt.figure(figsize=(8, 5))
            plt.plot(epochs, train_loss, label='Train')
            plt.plot(epochs, test_loss,  label='Test')
            plt.xlabel('Epoch')
            plt.ylabel('total_loss')
            #plt.title(f'{tree_name} Total Loss vs. Epoch')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.5)
            plt.show()

    # ====== 6) 特征相关矩阵（基于训练输入的特征）======
    X_train_used_df = pd.DataFrame(_as_array(X_train_used), columns=feat_names_used)
    corr = X_train_used_df.corr(numeric_only=True)
    corr = corr.dropna(axis=0, how='all').dropna(axis=1, how='all')
    plt.figure(figsize=(20, 20))
    plt.imshow(corr.values, aspect='equal', interpolation='none', vmin=-1, vmax=1, cmap='bwr')
    cbar = plt.colorbar(fraction=0.046, pad=0.04)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=10)
    plt.yticks(range(len(corr.index)),    corr.index, fontsize=10)
    #plt.title(f"{tree_name} Feature Correlation Matrix", fontsize=14)
    plt.tight_layout()
    plt.show()

    # ====== 7) 【去相关检验】仅此处使用“包含 decor 的 X_full” ======
    if len(decor_idx_full) == 0:
        print("[INFO] 未找到可用于去相关检验的变量（可能未指定或训练时已全部剔除）。")
        return

    decor_var_names = [full_feature_names[i] for i in decor_idx_full]

    # ---------- 7.1 【加权】相关系数矩阵：行=class score，列=decor 变量；分 Train/Test ----------
    def _build_corr_matrix(probs, Xmat_full, wvec):
        Xarr = _as_array(Xmat_full)
        wv = _safe_w(wvec)
        R = np.zeros((n_classes, len(decor_idx_full)), dtype=float)
        for r, ci in enumerate(range(n_classes)):
            s = probs[:, ci]
            for c, j in enumerate(decor_idx_full):
                x = Xarr[:, j]
                R[r, c] = _weighted_pearson(x, s, wv)
        return R

    R_train = _build_corr_matrix(probs_train, X_train_full, w_train)
    R_test  = _build_corr_matrix(probs_test,  X_test_full,  w_test)

    def _plot_corr_heatmap(R, title):
        plt.figure(figsize=(1.2 * len(decor_idx_full) + 6, 1.0 * n_classes + 4))
        plt.imshow(R, aspect='auto', interpolation='none', vmin=-1, vmax=1, cmap='bwr')
        cbar = plt.colorbar(fraction=0.046, pad=0.04, label="weighted Pearson r")
        cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
        plt.xticks(range(len(decor_var_names)), decor_var_names, rotation=45, ha='right', fontsize=11)
        plt.yticks(range(n_classes), class_names, fontsize=12)
        for i in range(n_classes):
            for j in range(len(decor_var_names)):
                val = R[i, j]
                plt.text(j, i, f"{val:+.2f}", ha='center', va='center',
                         color='white' if abs(val) > 0.5 else 'black', fontsize=10)
        #plt.title(title, fontsize=14)
        plt.tight_layout()
        plt.show()

    _plot_corr_heatmap(R_train, f"{tree_name} Decor-Check Heatmap (Train): class score vs decor")
    _plot_corr_heatmap(R_test,  f"{tree_name} Decor-Check Heatmap (Test): class score vs decor")

    # ---------- 7.2 【加权】decor 变量分布：每个 true class，用该 class 自己的 score 做“效率阈值”选择（Test） ----------
    try:
        X_test_arr = _as_array(X_test_full)
        y_all = np.asarray(y_test, int)
        w_all = _safe_w(w_test)

        eff_targets = np.array([1.0, 0.50, 0.1, 0.01], dtype=float)

        try:
            color_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", [])
        except Exception:
            color_cycle = []
        if len(color_cycle) < len(eff_targets):
            color_cycle = [f"C{i}" for i in range(len(eff_targets))]
        colors = color_cycle[:len(eff_targets)]

        for j, var_name in zip(decor_idx_full, decor_var_names):
            x_all = X_test_arr[:, j].astype(float)

            for class_idx, class_name in enumerate(class_names):
                mask_cls = (y_all == class_idx)
                if not np.any(mask_cls):
                    continue

                score_k = probs_test[:, class_idx].astype(float)
                score_cls = score_k[mask_cls]
                w_cls = w_all[mask_cls]

                if (score_cls.size == 0) or (w_cls.sum() <= 0) or (not np.any(np.isfinite(score_cls))):
                    continue

                thr_list = []
                for eff in eff_targets:
                    if eff >= 1.0 - 1e-12:
                        thr = -1e9
                    else:
                        q = 1.0 - eff
                        thr = float(_weighted_quantile(score_cls, [q], w_cls)[0])
                    thr_list.append(thr)

                x_cls = x_all[mask_cls]
                if not np.any(np.isfinite(x_cls)):
                    continue

                qlo, qhi = _weighted_quantile(x_cls, [0.01, 0.99], w_cls)
                if not np.isfinite(qlo) or not np.isfinite(qhi) or qlo == qhi:
                    qlo = np.nanmin(x_cls)
                    qhi = np.nanmax(x_cls)
                    if not np.isfinite(qlo) or not np.isfinite(qhi) or qlo == qhi:
                        continue

                span = float(qhi - qlo)
                xmin = float(qlo - 0.02 * span)
                xmax = float(qhi + 0.02 * span)
                n_bins = 100
                bin_edges = np.linspace(xmin, xmax, n_bins + 1)

                plt.figure(figsize=(8, 6))
                for thr, eff, c in zip(thr_list, eff_targets, colors):
                    sel = mask_cls & (score_k > thr) & np.isfinite(x_all) & np.isfinite(w_all)
                    if not np.any(sel):
                        continue
                    plt.hist(
                        x_all[sel],
                        bins=bin_edges,
                        weights=w_all[sel],
                        density=True,
                        histtype="step",
                        linewidth=2.0,
                        label=rf"{class_name} @ $\epsilon=${eff*100:g}%",
                        color=c,
                    )

                plt.xlabel(var_name)
                plt.ylabel(f"A.U.")
                plt.legend()
                plt.tight_layout()
                plt.show()

    except Exception as e:
        print(f"[WARN] decor 变量分布绘图时发生异常: {e}")

    # ---------- 7.3 【加权】散点图：对每个 class score vs 每个 decor 变量（Test） ----------
    def _scatter_var_vs_score_all_classes(Xmat_full, probs, wvec, var_idx, var_name):
        Xarr = _as_array(Xmat_full)
        x0 = Xarr[:, var_idx].astype(float)
        w0 = _safe_w(wvec)

        base_m = np.isfinite(x0) & np.isfinite(w0)
        if not np.any(base_m):
            return

        x = x0[base_m]
        w = w0[base_m]
        idx_base = np.nonzero(base_m)[0]

        n = len(x)
        max_n = 50000
        if n > max_n:
            p = w / (w.sum() + 1e-12)
            choose = np.random.RandomState(7).choice(np.arange(n), size=max_n, replace=False, p=p)
            idx_sel = idx_base[choose]
        else:
            idx_sel = idx_base

        x_s = x0[idx_sel]
        w_s = w0[idx_sel]

        qlo, qhi = _weighted_quantile(x_s, [0.01, 0.99], w_s)
        if np.isfinite(qlo) and np.isfinite(qhi) and qlo < qhi:
            span = float(qhi - qlo)
            xmin = float(qlo - 0.05 * span)
            xmax = float(qhi + 0.05 * span)
        else:
            xmin = np.nanmin(x_s)
            xmax = np.nanmax(x_s)
        if not np.isfinite(xmin) or not np.isfinite(xmax) or xmin == xmax:
            return

        for class_idx, class_name in enumerate(class_names):
            y0 = probs[:, class_idx].astype(float)
            y_s = y0[idx_sel]

            m = np.isfinite(x_s) & np.isfinite(y_s) & np.isfinite(w_s)
            if not np.any(m):
                continue
            xs = x_s[m]
            ys = y_s[m]
            ws = w_s[m]

            plt.figure(figsize=(7, 5))
            plt.scatter(
                xs, ys,
                s=np.clip(10.0 * (ws / (ws.max() + 1e-12)), 2, 30),
                alpha=0.25,
                edgecolors='none'
            )

            try:
                qs = np.linspace(0, 1, 21)
                edges = _weighted_quantile(xs, qs, ws)
                edges[0] -= 1e-12
                edges[-1] += 1e-12

                xc, yc = [], []
                for i in range(len(edges) - 1):
                    lo, hi = edges[i], edges[i + 1]
                    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
                        continue
                    mm = (xs >= lo) & (xs < hi) if i < len(edges) - 2 else (xs >= lo) & (xs <= hi)
                    if not np.any(mm):
                        continue
                    wsum = ws[mm].sum()
                    if wsum <= 1e-12:
                        continue
                    xc.append(0.5 * (lo + hi))
                    yc.append((ws[mm] * ys[mm]).sum() / (wsum + 1e-12))
                if xc:
                    plt.plot(xc, yc, linewidth=2.0, color="black")
            except Exception:
                pass

            plt.xlim(xmin, xmax)
            plt.xlabel(var_name)
            plt.ylabel(f"{class_name} score")
            #plt.title(f"{tree_name} Decor-Check Scatter (Test): {var_name} vs {class_name} score")
            plt.tight_layout()
            plt.show()

    for j, vname in zip(decor_idx_full, decor_var_names):
        _scatter_var_vs_score_all_classes(X_test_full, probs_test, w_test, j, vname)


In [ ]:
X2, y2, w2, X2_data, y2_data, w2_data = prepare_multi_data('fat2', branches2, ENTRIES_PER_SAMPLE * 2)

In [ ]:
thresholds = {
    'msd8_1': (40,  150),
    'msd8_2': (40,  150),
    'pt8_1' : (180, None),
    'pt8_2' : (180, None),
    'eta8_1' : (-2.4, 2.4),
    'eta8_2' : (-2.4, 2.4),
    #'sphereM' : (0, 900)
}
X2, y2, w2 = filter_X(X2, y2, w2, branches2, thresholds, apply_to_sentinel=True)
X2_data, y2_data, w2_data = filter_X(X2_data, y2_data, w2_data, branches2, thresholds, apply_to_sentinel=True)

In [ ]:
X2_std = standardize_X(X2)

In [ ]:
#check_weights(w2, "w2")
FEATURE_NAMES = list(X2_std.columns)
ix1 = X2_std.columns.get_loc("msd8_1")
ix2 = X2_std.columns.get_loc("msd8_2")
clf2, splits2 = train_multi_model_un(X2_std, y2, w2, 'bdt_fat2_v2.pkl', decorrelate_feature_names=[ix1])
#print(X2_std)

In [ ]:
plot_multi_results_un(clf2, splits2, 'fat2', decorrelate_feature_names=["msd8_1"])

In [ ]:
clf2, splits2 = train_multi_model(X2_std, y2, w2, 'bdt_v2.pkl')

In [ ]:
plot_multi_results(clf2, splits2, 'fat2')

In [ ]:
X3, y3, w3, X3_data, y3_data, w3_data = prepare_multi_data('fat3', branches3, ENTRIES_PER_SAMPLE)

In [ ]:
thresholds = {
    'msd8_1': (40,  150),
    'msd8_2': (40,  150),
    'msd8_3': (40,  150),
    'pt8_1' : (180, None),
    'pt8_2' : (180, None),
    'pt8_3' : (180, None),
    'eta8_1' : (-2.4, 2.4),
    'eta8_2' : (-2.4, 2.4),
    'eta8_3' : (-2.4, 2.4),
    #'sphereM' : (0, 1350)
}
X3, y3, w3 = filter_X(X3, y3, w3, branches3, thresholds, apply_to_sentinel=True)
X3_data, y3_data, w3_data = filter_X(X3_data, y3_data, w3_data, branches3, thresholds, apply_to_sentinel=True)

In [ ]:
X3_std = standardize_X(X3)

In [ ]:
FEATURE_NAMES = list(X3_std.columns)
ix1 = X3_std.columns.get_loc("msd8_1")
clf3, splits3 = train_multi_model_un(X3_std, y3, w3, 'bdt_5class_with_tagger_fat3_nocorr.pkl', decorrelate_feature_names=[ix1])

In [ ]:
plot_multi_results_un(clf3, splits3, 'fat3', decorrelate_feature_names=["sphereM"])

In [ ]:
plot_multi_results(clf3, splits3, 'fat3')

In [ ]:
clf3, splits3 = train_multi_model(X3_std, y3, w3, 'bdt_5class_with_tagger_fat3.pkl')

# Estimation

In [ ]:
def convert_weights(w_old: np.ndarray,
                    y: np.ndarray,
                    lumi: float,
                    channel: int,
                    test_size: float = 0.3,
                    isMulti: bool = False,
                    bkgRatio: float = 1.0) -> np.ndarray:
    """
    将原始权重 w_old 转换为用于显著度估算的权重：
      * background: w_new = w_old * (lumi / test_size) / 1e3
      *   signal   : w_new = w_old * (lumi / test_size) / 1e3 / scale

    其中 scale = sum(w_old[bkg]) / sum(w_old[sig])，使得在转换前 signal 与 background 的总权重相同。

    参数:
      w_old     - 原始每事件权重 (一维数组)
      y         - 标签数组 (1=signal, 0=background)
      lumi      - 积分亮度，单位 fb^-1
      test_size - 测试集比例 (train_test_split 的 test_size)
    返回:
      w_new     - 转换后的权重 (一维数组)
    """
    # 基础因子
    base = lumi / test_size / 1e5 * bkgRatio

    # 分开计算
    w_new = w_old * base
    # background
    mask_data = (y == -2)
    w_new[mask_data] = 1 #w_new[mask_data] * test_size
    # signal
    if not isMulti:
      scale = SCALE_2FAT if channel == 2 else SCALE_3FAT
      mask_sig = (y == 1) 
      w_new[mask_sig] = w_new[mask_sig] / scale  / bkgRatio
    else:
      scale_vvv = SCALE_2FAT_VVV if channel == 2 else SCALE_3FAT_VVV
      scale_vh  = SCALE_2FAT_VH  if channel == 2 else SCALE_3FAT_VH
      scale_vv = SCALE_2FAT_VV if channel == 2 else SCALE_3FAT_VV
      scale_tt  = SCALE_2FAT_TT  if channel == 2 else SCALE_3FAT_TT
      scale_qcd  = SCALE_2FAT_QCD  if channel == 2 else SCALE_3FAT_QCD
      mask_vvv = (y == 0)
      mask_vh  = (y == 1)
      mask_tt  = (y == 2)
      mask_vv  = (y == 3)
      mask_qcd = (y == 4)
      w_new[mask_vvv] = w_new[mask_vvv] / scale_vvv / bkgRatio
      w_new[mask_vh]  = w_new[mask_vh] / scale_vh / bkgRatio
      w_new[mask_vv] = w_new[mask_vv] / scale_vv
      w_new[mask_tt]  = w_new[mask_tt] / scale_tt
      w_new[mask_qcd]  = w_new[mask_qcd] / scale_qcd

    return w_new

In [ ]:
def find_optimal_significance(model,
                              X_test: np.ndarray,
                              y_test: np.ndarray,
                              w_old: np.ndarray,
                              lumi: float,
                              channel: int,
                              test_ratio: float = 0.3,
                              n_thresholds: int = 100,
                              bkgRatio: float = 1.0):
    """
    在给定模型和测试集上，扫描 BDT 分数阈值，找出使信号显著度最大的点。
    参数:
      model        - 已训练好的模型，需支持 predict_proba 或 decision_function
      X_test       - 测试集特征矩阵
      y_test       - 测试集标签 (1=signal, 0=background)
      w_old        - 测试集原始权重
      lumi         - 积分亮度，单位 fb^-1
      test_size    - 测试集比例，用于权重转换
      n_thresholds - 阈值扫描点数
    返回:
      result (dict) 包含以下键:
        'thresholds'        - 阈值数组
        'significances'     - 对应的显著度数组
        'signal_counts'     - 对应的 S 数组 (按新权重)
        'background_counts' - 对应的 B 数组
        'best_threshold'    - 最优阈值
        'best_significance' - 最优显著度
        'best_S'            - 最优时的信号权重和
        'best_B'            - 最优时的本底权重和
        （已有）
        'category_names'                       - 子信号类别名 ['www','wwz','wzz','zzz','wh','zh']
        'category_significances'               - 各子信号“信号=该类，背景=其余”的显著度
        'category_significance_errors'         - 上述显著度误差
        （新增）
        'category_best_S'                      - 各子信号在最优阈值处的 S_j（加权和）
        'category_best_S_err'                  - 各子信号在最优阈值处的 S_j 误差 (sqrt(sum w^2))
        'category_best_B'                      - 各子信号对应的“其余事件” B_j（加权和）
        'category_best_B_err'                  - 各子信号对应的 B_j 误差 (sqrt(sum w^2))
        'bkg_category_names'                   - 本底大类名 ['qcd','vv','ttbar']
        'bkg_category_best_B'                  - 各本底大类在最优阈值处的 B（加权和）
        'bkg_category_best_B_err'              - 各本底大类在最优阈值处的 B 误差 (sqrt(sum w^2))
        （已有）
        'best_signal_efficiency'               - S 最优点相对全体信号的效率
        'best_background_efficiency'           - B 最优点相对全体本底的效率
    """
    # 1. 计算新权重
    w = convert_weights(w_old, y_test, lumi, channel, test_ratio, bkgRatio=bkgRatio)

    # 2. 得到模型打分
    scores = model.predict_proba(X_test)[:, 1]

    # 3. 扫描阈值
    p = 0.005              # p<1，越小越在 1 附近密集；比如 0.5、0.3、0.2
    x = np.linspace(0, 1, n_thresholds)
    thresholds = x**p    # 幂变换
    significances = []
    S_list,   B_list   = [], []
    S_entry, B_entry   = [], [] 
    sigma_Ss, sigma_Bs = [], []
    sigs,     sig_errs = [], []

    for thr in thresholds:
        mask_sig = (y_test == 1) & (scores >= thr)
        mask_bkg = (y_test == 0) & (scores >= thr)

        w_sig = w[mask_sig]
        w_bkg = w[mask_bkg]

        # 加权和
        S = w_sig.sum()
        B = w_bkg.sum()

        # 原始条目
        S_e = mask_sig.sum()
        B_e = mask_bkg.sum()

        # 统计误差 (Poisson，考虑不同权重)
        sigma_S = np.sqrt((w_sig ** 2).sum())
        sigma_B = np.sqrt((w_bkg ** 2).sum())

        # 显著度
        if S + B > 0:
            Z = S / np.sqrt(S + B)

            # 误差传递
            denom32 = 2.0 * (S + B) ** 1.5
            dZ_dS = (S + 2 * B) / denom32
            dZ_dB = -S / denom32
            sigma_Z = np.sqrt((dZ_dS * sigma_S) ** 2 +
                              (dZ_dB * sigma_B) ** 2)
        else:
            Z = 0.0
            sigma_Z = 0.0

        # 收集
        S_list.append(S);          B_list.append(B)
        S_entry.append(S_e);       B_entry.append(B_e)
        sigma_Ss.append(sigma_S);  sigma_Bs.append(sigma_B)
        sigs.append(Z);            sig_errs.append(sigma_Z)

    # 转 numpy
    S_list   = np.asarray(S_list)
    B_list   = np.asarray(B_list)
    S_entry  = np.asarray(S_entry)
    B_entry  = np.asarray(B_entry)
    sigma_Ss = np.asarray(sigma_Ss)
    sigma_Bs = np.asarray(sigma_Bs)
    sigs     = np.asarray(sigs)
    sig_errs = np.asarray(sig_errs)

    # 4. 取最优阈值
    idx_opt = int(np.nanargmax(sigs))
    mask_pass = scores >= thresholds[idx_opt]

    # ========= 子信号按类别分解（含误差） =========
    sig_category_names = ['www', 'wwz', 'wzz', 'zzz', 'wh', 'zh']
    cat_Z, cat_dZ = [], []
    cat_S, cat_S_err = [], []
    cat_B, cat_B_err = [], []

    WEIGHT = WEIGHT_2FAT if channel == 2 else WEIGHT_3FAT
    SCALE  = SCALE_2FAT  if channel == 2 else SCALE_3FAT

    for name in sig_category_names:
        # 根据 w_old 区分样本
        if name == 'wh':
            mask_cat_old = (np.isclose(w_old, WEIGHT['wplush'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wminush'] * SCALE))
        elif name == 'zh':
            mask_cat_old = np.isclose(w_old, WEIGHT['zh'] * SCALE)
        else:
            mask_cat_old = np.isclose(w_old, WEIGHT[name.lower()] * SCALE)

        # 信号：既属于该类别，又通过阈值
        mask_Sj = mask_cat_old & mask_pass
        # 本底：通过阈值但不属于该类别（用于该类别的 Z_j 计算）
        mask_Bj = mask_pass & ~mask_cat_old

        w_Sj = w[mask_Sj];   w_Bj = w[mask_Bj]
        Sj   = w_Sj.sum();   Bj   = w_Bj.sum()
        sigma_Sj = np.sqrt((w_Sj**2).sum())
        sigma_Bj = np.sqrt((w_Bj**2).sum())

        if Sj + Bj > 0:
            Zj = Sj / np.sqrt(Sj + Bj)
            denom32_j = 2.0 * (Sj + Bj)**1.5
            dZ_dSj = (Sj + 2 * Bj) / denom32_j
            dZ_dBj = -Sj / denom32_j
            sigma_Zj = np.sqrt((dZ_dSj * sigma_Sj)**2 +
                               (dZ_dBj * sigma_Bj)**2)
        else:
            Zj, sigma_Zj = 0.0, 0.0

        cat_Z.append(Zj)
        cat_dZ.append(sigma_Zj)
        cat_S.append(Sj)
        cat_S_err.append(sigma_Sj)
        cat_B.append(Bj)
        cat_B_err.append(sigma_Bj)

    # ========= 本底大类（qcd / vv / ttbar）分解（含误差） =========
    # 利用权重映射构造各类遮罩；按最优阈值 + y_test==0 + 类遮罩 计数
    bkg_category_names = ['qcd', 'vv', 'ttbar']

    mask_qcd = (
        np.isclose(w_old, WEIGHT['qcd_ht100to200']) |
        np.isclose(w_old, WEIGHT['qcd_ht200to400']) |
        np.isclose(w_old, WEIGHT['qcd_ht400to600']) |
        np.isclose(w_old, WEIGHT['qcd_ht600to800']) |
        np.isclose(w_old, WEIGHT['qcd_ht800to1000']) |
        np.isclose(w_old, WEIGHT['qcd_ht1000to1200']) |
        np.isclose(w_old, WEIGHT['qcd_ht1200to1500']) |
        np.isclose(w_old, WEIGHT['qcd_ht1500to2000']) |
        np.isclose(w_old, WEIGHT['qcd_ht2000toinf'])
    )
    mask_vv = (
        np.isclose(w_old, WEIGHT['ww']) |
        np.isclose(w_old, WEIGHT['wz']) |
        np.isclose(w_old, WEIGHT['zz'])
    )
    mask_ttbar = (
        np.isclose(w_old, WEIGHT['tt_had']) |
        np.isclose(w_old, WEIGHT['tt_semilep'])
    )

    bkg_masks = {
        'qcd': mask_qcd,
        'vv': mask_vv,
        'ttbar': mask_ttbar,
    }

    bkg_B_vals, bkg_B_errs = [], []
    for name in bkg_category_names:
        m_old = bkg_masks[name]
        m = (y_test == 0) & mask_pass & m_old
        w_this = w[m]
        B_this = w_this.sum()
        B_this_err = np.sqrt((w_this**2).sum())
        bkg_B_vals.append(B_this)
        bkg_B_errs.append(B_this_err)

    # ========= （以下为原有绘图与返回） =========
    # 绘图
    plot_masks = {}

    # 先加入原有 category_names（vvv / vh）
    category_names = ['vvv', 'vh']
    for name in category_names:
        if name == 'vh':
            mask_cat_old = (np.isclose(w_old, WEIGHT['wplush'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wminush'] * SCALE) |
                            np.isclose(w_old, WEIGHT['zh'] * SCALE))
        elif name == 'vvv':
            mask_cat_old = (np.isclose(w_old, WEIGHT['www'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wwz'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wzz'] * SCALE) |
                            np.isclose(w_old, WEIGHT['zzz'] * SCALE))
        plot_masks[name] = mask_cat_old

    # 再加入 qcd / vv / ttbar
    plot_masks['qcd'] = mask_qcd
    plot_masks['vv'] = mask_vv
    plot_masks['ttbar'] = mask_ttbar

    bins = np.linspace(0.0, 1.0, 51)
    plt.figure()
    for label, m in plot_masks.items():
        if np.any(m):
            plt.hist(scores[m], bins=bins, range=(0, 1),
                     weights=w[m],
                     histtype='step', linewidth=2, label=label)
    plt.xlim(0, 1)
    plt.xlabel("BDT Score")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.ylim(1,None)
    plt.legend(ncol=1, fontsize=24)
    plt.tight_layout()
    plt.show()

    bins = np.linspace(0.98, 1, 51)
    plt.figure()
    for label, m in plot_masks.items():
        if np.any(m):
            plt.hist(scores[m], bins=bins, range=(0.98, 1),
                     weights=w[m],
                     histtype='step', linewidth=2, label=label)
    plt.xlim(0.98, 1)
    plt.xlabel("BDT Score")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.ylim(1,None)
    plt.legend(ncol=1, fontsize=24)
    plt.tight_layout()
    plt.show()

    S_total = w[y_test == 1].sum()
    B_total = w[y_test == 0].sum()
    best_S = S_list[idx_opt]
    best_B = B_list[idx_opt]
    best_signal_efficiency     = best_S / S_total if S_total > 0 else np.nan
    best_background_efficiency = best_B / B_total if B_total > 0 else np.nan

    # 返回时沿用你既有的 key 名称
    return {
        'thresholds':                     thresholds,
        'significances':                  sigs,
        'significance_errors':            sig_errs,
        'signal_counts':                  S_list,
        'background_counts':              B_list,
        'signal_errors':                  sigma_Ss,
        'background_errors':              sigma_Bs,
        'signal_entries':                 S_entry,
        'background_entries':             B_entry,
        'best_threshold':                 thresholds[idx_opt],
        'best_significance':              sigs[idx_opt],
        'best_significance_error':        sig_errs[idx_opt],
        'best_S':                         S_list[idx_opt],
        'best_B':                         B_list[idx_opt],
        'best_S_entry':                   S_entry[idx_opt],
        'best_B_entry':                   B_entry[idx_opt],

        # 子信号类别（与此前保持一致的 key）
        'category_names':                 sig_category_names,
        'category_significances':         np.array(cat_Z),
        'category_significance_errors':   np.array(cat_dZ),
        'category_best_S':                np.array(cat_S),
        'category_best_S_err':            np.array(cat_S_err),   # 新增误差
        'category_best_B':                np.array(cat_B),
        'category_best_B_err':            np.array(cat_B_err),   # 新增误差

        # 本底大类分解（新增）
        'bkg_category_names':             np.array(bkg_category_names),
        'bkg_category_best_B':            np.array(bkg_B_vals),
        'bkg_category_best_B_err':        np.array(bkg_B_errs),

        'best_signal_efficiency':         best_signal_efficiency,
        'best_background_efficiency':     best_background_efficiency,
    }


In [ ]:
def find_optimal_significance_combine(
    model,
    X_test,
    y_test,
    w_old,
    lumi,
    channel,
    test_ratio: float = 0.3,
    n_thresholds: int = 4,
    bkgRatio: float = 1.0,
    topN: int = 4,
    min_bkg_weight: float = 5.0,
    manual_bins=None,              # 此版本忽略 manual_bins
    nClass: int = 5,               # 默认5类：扫描前四个维度 VVV,VH,TT,VV
    debug: bool = False,
    debug_topk: int = 10,
):

    # ---------- 调试工具 ----------
    def _fmt(v, nd=6):
        try:
            if np.isscalar(v): return f"{float(v):.{nd}f}"
            arr = np.asarray(v).ravel()
            return "[" + ", ".join(f"{float(x):.{nd}f}" for x in arr) + "]"
        except Exception:
            return str(v)
    def _p(msg, indent=0):
        if debug: print("  " * indent + str(msg))
    def _key_from_bin(lows_vec, highs_vec, nd=12):
        lv = tuple(float(np.round(v, nd)) for v in lows_vec)
        hv = tuple(float(np.round(v, nd)) for v in highs_vec)
        return (lv, hv)

    # 判断两个轴对齐超矩形是否重叠（左闭右开）
    def _overlap(lo1, hi1, lo2, hi2, eps=1e-12):
        for d in range(len(lo1)):
            if not (lo1[d] < hi2[d] - eps and lo2[d] < hi1[d] - eps):
                return False
        return True

    # ---------- 0) 权重换算 & 概率提取 ----------
    w = convert_weights(w_old, y_test, lumi, channel, test_ratio, isMulti=True, bkgRatio=bkgRatio)

    NAME_BY_LABEL = ["VVV", "VH", "TT", "VV", "QCD"]
    m_VVV = (y_test == 0)
    m_VH  = (y_test == 1)
    m_TT  = (y_test == 2)
    m_VV  = (y_test == 3)
    m_QCD = (y_test == 4)

    proba = model.predict_proba(X_test)
    if proba.ndim == 1:
        proba = np.stack([1.0 - proba, proba], axis=1)
    n, ncols = proba.shape

    # 类索引映射
    cls_idx = {}
    if hasattr(model, "classes_"):
        for lab, name in enumerate(NAME_BY_LABEL):
            idx_arr = np.where(model.classes_ == lab)[0]
            if len(idx_arr) == 1:
                cls_idx[name] = int(idx_arr[0])
    if len(cls_idx) < 5 and ncols >= 5:
        cls_idx = {name: i for i, name in enumerate(NAME_BY_LABEL)}

    # 维度数：2→1D，3→2D，4→3D，5+→4D
    if nClass <= 2:   D = 1
    elif nClass == 3: D = 2
    elif nClass == 4: D = 3
    else:             D = 4

    # 扫描轴（最多4维：VVV,VH,TT,VV）
    if D == 1:
        if ncols >= 5 and len(cls_idx) == 5:
            s_sig = proba[:, cls_idx["VVV"]] + proba[:, cls_idx["VH"]]
        elif ncols == 2:
            s_sig = proba[:, 1]
        else:
            s_sig = proba[:, -1]
        score_axes = [s_sig]
        axis_names = ["sigScore(VVV+VH)"]
    else:
        need = ["VVV", "VH", "TT", "VV"][:D]
        if ncols < 5 or any(k not in cls_idx for k in need):
            raise ValueError(f"nClass={nClass} 需要多类概率(>=5列)，并包含 {need}。")
        score_axes = [proba[:, cls_idx[k]] for k in need]
        axis_names = [f"p({k})" for k in need]

    # 信号/本底总体
    is_sig = (m_VVV | m_VH)
    is_bkg = (m_TT | m_VV | m_QCD)
    S_total = float(w[is_sig].sum())
    B_total = float(w[is_bkg].sum())

    _p(f"[初始化] 维度D={D}, 轴={axis_names}")
    _p(f"[初始化] S_total={S_total:.3f}, B_total={B_total:.3f}")

    # ---------- 1) 每维的 1D 尾部扫描（用于返回与参考） ----------
    def _calc_Z_from_SB(S, B, sS, sB):
        if S + B <= 0: return 0.0, 0.0
        Z = S / np.sqrt(S + B)
        denom32 = 2.0 * (S + B)**1.5
        dZ_dS = (S + 2*B) / denom32
        dZ_dB = -S / denom32
        sZ = np.sqrt((dZ_dS*sS)**2 + (dZ_dB*sB)**2)
        return float(Z), float(sZ)

    p_exp = 0.005
    thr_1d = np.linspace(0.0, 1.0, int(max(2, n_thresholds)))
    thr_1d = np.clip(thr_1d**p_exp, 0.0, 1.0)

    T = len(thr_1d)
    S_tail_by_dim = np.zeros((D, T), dtype=float)
    B_tail_by_dim = np.zeros((D, T), dtype=float)
    S_ent_by_dim = np.zeros((D, T), dtype=float)
    B_ent_by_dim = np.zeros((D, T), dtype=float)
    sS_tail_by_dim = np.zeros((D, T), dtype=float)
    sB_tail_by_dim = np.zeros((D, T), dtype=float)
    Z_tail_by_dim  = np.zeros((D, T), dtype=float)
    sZ_tail_by_dim = np.zeros((D, T), dtype=float)

    for d in range(D):
        s = score_axes[d] if D > 1 else score_axes[0]
        for it, thr in enumerate(thr_1d):
            m_bin = (s >= thr)
            wS = w[m_bin & is_sig]; wB = w[m_bin & is_bkg]
            S = wS.sum(); B = wB.sum()
            sS = np.sqrt((wS**2).sum()); sB = np.sqrt((wB**2).sum())
            Se = (m_bin & is_sig).sum(); Be = (m_bin & is_bkg).sum()
            Z, sZ = _calc_Z_from_SB(S, B, sS, sB)
            S_tail_by_dim[d, it] = S;  B_tail_by_dim[d, it] = B
            sS_tail_by_dim[d, it] = sS; sB_tail_by_dim[d, it] = sB
            S_ent_by_dim[d, it] = Se;  B_ent_by_dim[d, it] = Be
            Z_tail_by_dim [d, it] = Z; sZ_tail_by_dim[d, it] = sZ

    # ---------- 2) 评估工具 ----------
    def _build_tails_masked(mask_sel, edges):
        arrs = [score_axes[d][mask_sel] for d in range(D)]
        Hw = np.histogramdd(tuple(arrs), bins=edges, weights=w[mask_sel])[0]
        def tail_from_high(A):
            T = A.copy()
            for ax in range(D):
                T = np.flip(T, axis=ax); T = np.cumsum(T, axis=ax); T = np.flip(T, axis=ax)
            return T
        return tail_from_high(Hw)

    def _rect_sum_from_tail(tail, lows_idx, highs_idx):
        tot = 0.0
        for bits in range(1 << D):
            idx = []; pop = 0
            for d in range(D):
                if (bits >> d) & 1:
                    idx.append(int(highs_idx[d])); pop += 1
                else:
                    idx.append(int(lows_idx[d]))
            tot += (-1.0 if (pop & 1) else 1.0) * tail[tuple(idx)]
        return float(tot)

    # ============= 关键改动开始：非重叠 + “一维slice + 其余尾部”候选 =============
    selected_bins = []  # 保存已选 bin：元素为 (low_vec, high_vec)
    unit_high = [1.0 for _ in range(D)]  # 全局上界恒为 1

    top_bins = []
    for k in range(topN):
        _p(f"\n===== 选择第 {k+1}/{topN} 个 bin =====")
        _p(f"[开始] 当前高端 H={_fmt(unit_high)}")
        # —— 累计组合池（跨轮累计） —— #
        accum_map = {}

        # --- 5 轮逐维细化 ---
        ROUNDS = 5
        cand_vals = [None] * D
        for d in range(D):
            cand_vals[d] = list(np.linspace(0.0, unit_high[d], int(max(2, n_thresholds)), endpoint=False))
        _p(f"[轮0] 初始候选 per-dim: " + ", ".join(f"{len(cand_vals[d])}" for d in range(D)), 1)

        for r in range(ROUNDS):
            _p(f"[轮{r+1}] 构造 edges 与 tails", 1)
            # 1) edges / 映射
            edges = []; base_lists = []; val2idx = []; highs_idx = []
            for d in range(D):
                base = np.unique(np.r_[cand_vals[d], unit_high[d]])
                edges.append(np.r_[base, base[-1] + 1e-9])
                base_lists.append(base)
                val2idx.append({float(v): i for i, v in enumerate(base)})
                highs_idx.append(int(val2idx[-1][float(unit_high[d])]))
                _p(f"  维{d} 候选={len(cand_vals[d])}, 区间=[0,{unit_high[d]:.6f}) 示例={_fmt(base[:min(5,len(base))])}...", 2)

            # 2) 构建 S/B tail（带掩码）
            S_tailD = _build_tails_masked(is_sig, edges)
            B_tailD = _build_tails_masked(is_bkg, edges)

            # 3) 按“一维slice + 其余尾部”生成候选并评估
            low_idx_lists = []
            for d in range(D):
                idxs = [val2idx[d].get(float(v), None) for v in cand_vals[d]]
                idxs = [int(i) for i in idxs if i is not None and i < highs_idx[d]]
                if len(idxs) == 0: idxs = [0]
                low_idx_lists.append(sorted(set(idxs)))

            combos_round = []
            # 允许 exactly-one-slice：在每个维度各自当 slice 一次
            for slice_dim in range(D):
                for lows in product(*low_idx_lists):
                    lows = list(lows)
                    # 构造对应的 highs：slice_dim 用 i+1，其余维用全局 high
                    hi_idx_vec = []
                    valid = True
                    for d in range(D):
                        if d == slice_dim:
                            li = lows[d]
                            if li + 1 <= highs_idx[d]:
                                hi_idx_vec.append(li + 1)
                            else:
                                valid = False; break
                        else:
                            hi_idx_vec.append(highs_idx[d])
                    if not valid:
                        continue

                    # 计算 S/B/Z
                    S_bin_tail = _rect_sum_from_tail(S_tailD, lows, hi_idx_vec)
                    B_bin_tail = _rect_sum_from_tail(B_tailD, lows, hi_idx_vec)
                    Z_tail = S_bin_tail / np.sqrt(S_bin_tail + B_bin_tail) if (S_bin_tail + B_bin_tail) > 0 else 0.0

                    thr_low_vals  = [float(base_lists[d][lows[d]]) for d in range(D)]
                    thr_high_vals = []
                    for d in range(D):
                        if d == slice_dim:
                            thr_high_vals.append(float(base_lists[d][lows[d] + 1]))
                        else:
                            thr_high_vals.append(float(unit_high[d]))

                    # —— 非重叠过滤（与已选 bin 不重叠才纳入池）——
                    non_overlap = True
                    for (lo_sel, hi_sel) in selected_bins:
                        if _overlap(thr_low_vals, thr_high_vals, lo_sel, hi_sel):
                            non_overlap = False
                            break
                    if not non_overlap:
                        continue

                    combos_round.append((Z_tail, thr_low_vals, thr_high_vals,
                                         float(S_bin_tail), float(B_bin_tail)))

            combos_round.sort(key=lambda x: x[0], reverse=True)

            _p(f"[轮{r+1}] 本轮组合总数={len(combos_round)}，min_bkg={min_bkg_weight}", 1)
            for rank, (Z_tail, lows_vals, highs_vals, S_est, B_est) in enumerate(combos_round[:debug_topk], start=1):
                _p(f"  RoundTop{rank}: Z={Z_tail:.4f}  S={S_est:.3f}  B={B_est:.3f}  "
                   f"lows={_fmt(lows_vals)} highs={_fmt(highs_vals)}", 2)

            # —— 合并进累计池（key 用 lows+highs，去重；保留更大Z） —— #
            for rank, (Z_tail, lows_vals, highs_vals, S_est, B_est) in enumerate(combos_round, start=1):
                key = _key_from_bin(lows_vals, highs_vals)
                if (key not in accum_map) or (Z_tail > accum_map[key]["Z"]):
                    accum_map[key] = {
                        "Z": Z_tail, "S": S_est, "B": B_est,
                        "lows": tuple(lows_vals), "highs": tuple(highs_vals),
                        "round": r+1, "rank_in_round": rank
                    }

            # —— 用“累计池”的排序来抽逐维 top-3 阈值，驱动下一轮细化 —— #
            accum_sorted = sorted(accum_map.values(), key=lambda d: d["Z"], reverse=True)
            _p(f"[轮{r+1}] 累计组合总数={len(accum_sorted)}", 1)
            for rank, item in enumerate(accum_sorted[:debug_topk], start=1):
                _p(f"  AccTop{rank}: Z={item['Z']:.4f}  S={item['S']:.3f}  B={item['B']:.3f}  "
                   f"lows={_fmt(item['lows'])} highs={_fmt(item['highs'])}  "
                   f"(from round#{item['round']} rrank={item['rank_in_round']})", 2)

            pick_per_dim = [ [] for _ in range(D) ]
            pick_info    = [ [] for _ in range(D) ]
            seen_per_dim = [ set() for _ in range(D) ]
            for gl_rank, item in enumerate(accum_sorted, start=1):
                lows_vals = item["lows"]; Z_tail = item["Z"]; S_est = item["S"]; B_est = item["B"]
                for d in range(D):
                    v = float(lows_vals[d])
                    if v not in seen_per_dim[d]:
                        seen_per_dim[d].add(v)
                        pick_per_dim[d].append(v)
                        pick_info[d].append((v, Z_tail, S_est, B_est, gl_rank, item["round"], item["rank_in_round"]))
                if all(len(pick_per_dim[d]) >= 3 for d in range(D)):
                    break

            for d in range(D):
                _p(f"[轮{r+1}] 维{d}（基于累计）抽到 top-3 阈值：", 1)
                for (v, Z_tail, S_est, B_est, gr, rr, rrank) in pick_info[d][:3]:
                    _p(f"  thr={v:.6f}  (AccRank={gr}, Round#{rr}/Rank{rrank})  Z={Z_tail:.4f}  S={S_est:.3f}  B={B_est:.3f}", 2)

            # 下一轮细化区间：用每维 pick 的[min,max)均分
            new_cands = []
            for d in range(D):
                picks = sorted(set(pick_per_dim[d])) if len(pick_per_dim[d]) else [0.0, unit_high[d]]
                lo_d = float(max(0.0, min(picks)))
                hi_d = float(min(unit_high[d], max(picks)))
                if hi_d <= lo_d + 1e-12:
                    span = max(1e-3, 0.1 * unit_high[d])
                    lo_d = max(0.0, unit_high[d] - span)
                    hi_d = unit_high[d]
                new = np.linspace(lo_d, hi_d, int(max(2, n_thresholds)), endpoint=False)
                new_cands.append(list(new))
                _p(f"[轮{r+1}] 维{d} 下一轮细化区间 [{lo_d:.6f},{hi_d:.6f}) 均分 {len(new)} 个", 1)
            cand_vals = new_cands

        # --- 最终选择：从“累计池”里选 非重叠 且 B≥阈值 的最大Z --- #
        accum_sorted = sorted(accum_map.values(), key=lambda d: d["Z"], reverse=True)
        _p(f"[最终选择前] 累计组合总数={len(accum_sorted)}，min_bkg={min_bkg_weight}", 1)
        for rank, item in enumerate(accum_sorted[:debug_topk], start=1):
            ok_b = (item["B"] >= float(min_bkg_weight))
            ok_ol = all(not _overlap(item["lows"], item["highs"], lo_sel, hi_sel)
                        for (lo_sel, hi_sel) in selected_bins)
            status = ("PASS" if ok_b and ok_ol else
                      "FAIL_BKG" if not ok_b else "FAIL_OLAP")
            _p(f"  AccTop{rank}: Z={item['Z']:.4f}  S={item['S']:.3f}  B={item['B']:.3f}({status})  "
               f"lows={_fmt(item['lows'])} highs={_fmt(item['highs'])}  "
               f"(from round#{item['round']} rrank={item['rank_in_round']})", 2)

        chosen = None
        for item in accum_sorted:
            if item["B"] >= float(min_bkg_weight) and \
               all(not _overlap(item["lows"], item["highs"], lo_sel, hi_sel)
                   for (lo_sel, hi_sel) in selected_bins):
                chosen = item
                break
        if chosen is None:
            _p("没有任何组合满足(非重叠 & 本底阈值)，提前结束该 bin。", 1)
            break

        thr_low_vec  = list(map(float, chosen["lows"]))
        thr_high_vec = list(map(float, chosen["highs"]))
        _p(f"[选中] Z={chosen['Z']:.4f}  S={chosen['S']:.3f}  B={chosen['B']:.3f}", 1)
        _p(f"[选中] lows={_fmt(thr_low_vec)}  highs={_fmt(thr_high_vec)}", 1)

        # === 精确事件级统计 ===
        def _mask_dim(d, lo, hi):
            s = score_axes[d]
            return ((s >= lo) & (s < hi)) if hi < 1.0 - 1e-12 else (s >= lo)
        m_bin = np.ones(n, dtype=bool)
        for d in range(D):
            m_bin &= _mask_dim(d, thr_low_vec[d], thr_high_vec[d])

        wS = w[m_bin & is_sig];  wB = w[m_bin & is_bkg]
        S_bin = float(wS.sum());  B_bin = float(wB.sum())
        sS_bin = float(np.sqrt((wS**2).sum()))
        sB_bin = float(np.sqrt((wB**2).sum()))
        S_e = int((m_bin & is_sig).sum()); B_e = int((m_bin & is_bkg).sum())
        Z_bin, sZ_bin = _calc_Z_from_SB(S_bin, B_bin, sS_bin, sB_bin)
        _p(f"[精确统计] Z={Z_bin:.4f}±{sZ_bin:.4f}  S={S_bin:.3f}±{sS_bin:.3f}  B={B_bin:.3f}±{sB_bin:.3f}", 1)

        # === 记录 & “只扣掉该bin” ===
        selected_bins.append((thr_low_vec[:], thr_high_vec[:]))  # 仅记录，不修改各维高端
        _p(f"[扣掉bin] 已选 {len(selected_bins)} 个：", 1)
        for i, (lo_i, hi_i) in enumerate(selected_bins, 1):
            _p(f"  sel#{i}: low={_fmt(lo_i)}  high={_fmt(hi_i)}", 2)

        # -------- 五类分解（这里是 BUGFIX 部分） --------
        # 预先计算：这个 bin 内的总权重和 w^2 总和
        W_bin  = S_bin + B_bin                      # = sum_{m_bin} w
        w2_bin = sS_bin**2 + sB_bin**2              # = sum_{m_bin} w^2

        cat_Z, cat_sZ, cat_S, cat_sS, cat_B, cat_sB = [], [], [], [], [], []
        for lab, name in enumerate(NAME_BY_LABEL):
            # 该类在 bin 内的“信号”
            mC = (y_test == lab) & m_bin
            wC = w[mC]
            S_j = float(wC.sum())
            w2_j = float((wC**2).sum())
            sS_j = float(np.sqrt(w2_j))

            # —— BUGFIX：B(rest)_j 不再直接用 w[m_bin & (y_test != lab)] 求和 ——
            # 在同一个 bin 内，“其余事件”的权重就是 总权重 - S_j
            B_j = W_bin - S_j

            # 对应的误差：rest 的 w^2 总和 = bin 的 w^2 总和 - 该类 w^2
            w2_rest = max(0.0, w2_bin - w2_j)
            sB_j = float(np.sqrt(w2_rest))

            Z_j, sZ_j = _calc_Z_from_SB(S_j, B_j, sS_j, sB_j)
            cat_Z.append(Z_j);   cat_sZ.append(sZ_j)
            cat_S.append(S_j);   cat_sS.append(sS_j)
            cat_B.append(B_j);   cat_sB.append(sB_j)

        bkg_names = np.array(["qcd", "vv", "ttbar"])
        bkg_B_vals = np.array([
            float(w[m_bin & m_QCD].sum()),
            float(w[m_bin & m_VV ].sum()),
            float(w[m_bin & m_TT ].sum()),
        ], dtype=float)
        bkg_B_errs = np.array([
            float(np.sqrt((w[m_bin & m_QCD]**2).sum())),
            float(np.sqrt((w[m_bin & m_VV ]**2).sum())),
            float(np.sqrt((w[m_bin & m_TT ]**2).sum())),
        ], dtype=float)

        # 效率
        bin_sig_eff = (S_bin / S_total) if S_total > 0 else np.nan
        bin_bkg_eff = (B_bin / B_total) if B_total > 0 else np.nan

        # 尾部效率近似（按各维 thr_1d 就近点）
        tail_sig_eff = []; tail_bkg_eff = []
        for d in range(D):
            idx = int(np.searchsorted(thr_1d, thr_low_vec[d], side="right") - 1)
            idx = max(0, min(idx, T - 1))
            tail_S = S_tail_by_dim[d, idx]; tail_B = B_tail_by_dim[d, idx]
            tail_sig_eff.append((tail_S / S_total) if S_total > 0 else np.nan)
            tail_bkg_eff.append((tail_B / B_total) if B_total > 0 else np.nan)

        top_bins.append({
            "bin_index": len(top_bins) + 1,
            "thr_low":  np.array(thr_low_vec,  dtype=float),
            "thr_high": np.array(thr_high_vec, dtype=float),
            "significance": float(Z_bin),
            "significance_error": float(sZ_bin),
            "S": S_bin, "S_err": sS_bin, "S_entries": int(S_e),
            "B": B_bin, "B_err": sB_bin, "B_entries": int(B_e),
            "category_names": np.array(NAME_BY_LABEL),
            "category_significances": np.array(cat_Z,  dtype=float),
            "category_significance_errors": np.array(cat_sZ, dtype=float),
            "category_S":  np.array(cat_S,  dtype=float),
            "category_S_err": np.array(cat_sS, dtype=float),
            "category_B":  np.array(cat_B,  dtype=float),
            "category_B_err": np.array(cat_sB, dtype=float),
            "bkg_category_names": bkg_names,
            "bkg_B": bkg_B_vals,
            "bkg_B_err": bkg_B_errs,
            "bin_signal_efficiency": float(bin_sig_eff),
            "bin_background_efficiency": float(bin_bkg_eff),
            "tail_signal_efficiency": np.array(tail_sig_eff, dtype=float),
            "tail_background_efficiency": np.array(tail_bkg_eff, dtype=float),
        })
    # ============= 关键改动结束 =============

    n_selected_bins_only_topN = len(top_bins)

    # ---------- 4) 合并显著度 ----------
    if len(top_bins) > 0:
        S_sum  = float(np.sum([b["S"]     for b in top_bins]))
        B_sum  = float(np.sum([b["B"]     for b in top_bins]))
        sS_sum = float(np.sqrt(np.sum([b["S_err"]**2 for b in top_bins])))
        sB_sum = float(np.sqrt(np.sum([b["B_err"]**2 for b in top_bins])))
        Z_vec  = np.array([b["significance"] for b in top_bins], dtype=float)
        sZ_vec = np.array([b["significance_error"] for b in top_bins], dtype=float)
        Z_comb = float(np.sqrt(np.sum(Z_vec**2)))
        sZ_comb = float(np.sqrt(np.sum((Z_vec**2) * (sZ_vec**2))) / Z_comb) if Z_comb > 0 else float(np.sqrt(np.sum(sZ_vec**2)))
    else:
        S_sum = B_sum = sS_sum = sB_sum = 0.0
        Z_comb = sZ_comb = 0.0

    # ---------- 5) best_* ----------
    if len(top_bins) > 0:
        best_bin = top_bins[0]
        best_threshold = best_bin["thr_low"]
        best_S = best_bin["S"]; best_B = best_bin["B"]
        best_S_entry = best_bin["S_entries"]; best_B_entry = best_bin["B_entries"]
        best_Z = best_bin["significance"];   best_Z_err  = best_bin["significance_error"]
        best_sig_eff = best_bin["bin_signal_efficiency"]
        best_bkg_eff = best_bin["bin_background_efficiency"]
        best_tail_sig_eff = best_bin["tail_signal_efficiency"]
        best_tail_bkg_eff = best_bin["tail_background_efficiency"]

        cat_Z   = best_bin["category_significances"]
        cat_dZ  = best_bin["category_significance_errors"]
        cat_S   = best_bin["category_S"];     cat_S_err = best_bin["category_S_err"]
        cat_B   = best_bin["category_B"];     cat_B_err = best_bin["category_B_err"]
        bkg_names = best_bin["bkg_category_names"]
        bkg_B_vals = best_bin["bkg_B"];       bkg_B_errs = best_bin["bkg_B_err"]
    else:
        best_threshold = np.array([1.0] * D, dtype=float)
        best_S = best_B = best_S_entry = best_B_entry = 0.0
        best_Z = best_Z_err = 0.0
        best_sig_eff = best_bkg_eff = np.nan
        best_tail_sig_eff = np.array([np.nan]*D, dtype=float)
        best_tail_bkg_eff = np.array([np.nan]*D, dtype=float)
        cat_Z = cat_dZ = cat_S = cat_S_err = cat_B = cat_B_err = np.array([])
        bkg_names = np.array(["qcd", "vv", "ttbar"])
        bkg_B_vals = bkg_B_errs = np.array([0.0, 0.0, 0.0])

    # ---------- 6) 绘图：VVV/VH 0–1 ----------
    bins = np.linspace(0.0, 1.0, 101)
    if ncols >= 5 and all(k in cls_idx for k in ["VVV", "VH"]):
        pVVV = proba[:, cls_idx["VVV"]]
        pVH  = proba[:, cls_idx["VH"]]
        plt.figure()
        for lab, name in enumerate(NAME_BY_LABEL):
            m = (y_test == lab)
            plt.hist(pVVV[m], bins=bins, weights=w[m], histtype="step", linewidth=2, label=name)
        plt.xlim(0, 1); plt.yscale("log"); plt.ylim(1e-2, None)
        plt.xlabel("VVV score"); plt.ylabel("Events"); plt.legend(fontsize=16)
        plt.tight_layout(); plt.show()
        plt.figure()
        for lab, name in enumerate(NAME_BY_LABEL):
            m = (y_test == lab)
            plt.hist(pVH[m], bins=bins, weights=w[m], histtype="step", linewidth=2, label=name)
        plt.xlim(0, 1); plt.yscale("log"); plt.ylim(1e-2, None)
        plt.xlabel("VH score"); plt.ylabel("Events"); plt.legend(fontsize=16)
        plt.tight_layout(); plt.show()
    else:
        s_plot = score_axes[0]
        plt.figure()
        for lab, name in enumerate(NAME_BY_LABEL):
            m = (y_test == lab)
            plt.hist(s_plot[m], bins=bins, weights=w[m], histtype="step", linewidth=2, label=name)
        plt.xlim(0, 1); plt.yscale("log"); plt.ylim(1e-2, None)
        plt.xlabel("sigScore(VVV+VH)"); plt.ylabel("Events"); plt.legend(fontsize=16)
        plt.tight_layout(); plt.show()

    bins = np.linspace(0., 0.1, 101)
    if ncols >= 5 and all(k in cls_idx for k in ["VVV", "VH"]):
        pVVV = proba[:, cls_idx["VVV"]]
        pVH  = proba[:, cls_idx["VH"]]
        plt.figure()
        for lab, name in enumerate(NAME_BY_LABEL):
            m = (y_test == lab)
            plt.hist(pVVV[m], bins=bins, weights=w[m], histtype="step", linewidth=2, label=name)
        plt.xlim(0., 0.1); plt.yscale("log"); plt.ylim(1e-3, None)
        plt.xlabel("VVV score"); plt.ylabel("Events"); plt.legend(fontsize=16)
        plt.tight_layout(); plt.show()
        plt.figure()
        for lab, name in enumerate(NAME_BY_LABEL):
            m = (y_test == lab)
            plt.hist(pVH[m], bins=bins, weights=w[m], histtype="step", linewidth=2, label=name)
        plt.xlim(0., 0.1); plt.yscale("log"); plt.ylim(1e-3, None)
        plt.xlabel("VH score"); plt.ylabel("Events"); plt.legend(fontsize=16)
        plt.tight_layout(); plt.show()
    else:
        s_plot = score_axes[0]
        plt.figure()
        for lab, name in enumerate(NAME_BY_LABEL):
            m = (y_test == lab)
            plt.hist(s_plot[m], bins=bins, weights=w[m], histtype="step", linewidth=2, label=name)
        plt.xlim(0., 0.1); plt.yscale("log"); plt.ylim(1e-3, None)
        plt.xlabel("sigScore(VVV+VH)"); plt.ylabel("Events"); plt.legend(fontsize=16)
        plt.tight_layout(); plt.show()

    # ---------- 7) 返回 ----------
    return {
        "thresholds":                  thr_1d if D == 1 else np.vstack([thr_1d for _ in range(D)]),
        "significances":               Z_tail_by_dim[0] if D == 1 else Z_tail_by_dim,
        "significance_errors":         sZ_tail_by_dim[0] if D == 1 else sZ_tail_by_dim,
        "signal_counts":               S_tail_by_dim[0] if D == 1 else S_tail_by_dim,
        "background_counts":           B_tail_by_dim[0] if D == 1 else B_tail_by_dim,
        "signal_errors":               sS_tail_by_dim[0] if D == 1 else sS_tail_by_dim,
        "background_errors":           sB_tail_by_dim[0] if D == 1 else sB_tail_by_dim,
        "signal_entries":              S_ent_by_dim[0] if D == 1 else S_ent_by_dim,
        "background_entries":          B_ent_by_dim[0] if D == 1 else B_ent_by_dim,

        "best_threshold":              best_threshold,
        "best_significance":           float(best_Z),
        "best_significance_error":     float(best_Z_err),
        "best_S":                      float(best_S),
        "best_B":                      float(best_B),
        "best_S_entry":                int(best_S_entry),
        "best_B_entry":                int(best_B_entry),

        "category_names":              np.array(NAME_BY_LABEL),
        "category_significances":      np.array(cat_Z),
        "category_significance_errors":np.array(cat_dZ),
        "category_best_S":             np.array(cat_S),
        "category_best_S_err":         np.array(cat_S_err),
        "category_best_B":             np.array(cat_B),
        "category_best_B_err":         np.array(cat_B_err),

        "bkg_category_names":          bkg_names,
        "bkg_category_best_B":         bkg_B_vals,
        "bkg_category_best_B_err":     bkg_B_errs,

        "best_signal_efficiency":      float(best_sig_eff),
        "best_background_efficiency":  float(best_bkg_eff),
        "best_tail_signal_efficiency": np.array(best_tail_sig_eff, dtype=float),
        "best_tail_background_efficiency": np.array(best_tail_bkg_eff, dtype=float),

        "top_bins":                    top_bins,
        "n_selected_bins":             int(len(top_bins)),

        "combined_topN_S":             float(S_sum),
        "combined_topN_S_err":         float(sS_sum),
        "combined_topN_B":             float(B_sum),
        "combined_topN_B_err":         float(sB_sum),
        "combined_topN_significance":  float(Z_comb),
        "combined_topN_significance_error": float(sZ_comb),
    }


In [ ]:
def find_optimal_significance_multi(model,
                              X_test: np.ndarray,
                              y_test: np.ndarray,
                              w_old: np.ndarray,
                              lumi: float,
                              channel: int,
                              test_ratio: float = 0.3,
                              n_thresholds: int = 100):
    """
    多分类适配版：在给定模型和测试集上，二维扫描 (thr_vvv, thr_vh) 阈值，
    选择条件： p(VVV) >= thr_vvv  或  p(VH) >= thr_vh
    S = 通过选择的 (真) VVV 或 VH 的权重和
    B = 通过选择的 (真) BKG 的权重和
    显著度 Z = S / sqrt(S + B)

    参数与返回的键名保持与原实现一致：
      thresholds:      一维阈值数组（两轴共用）
      significances:   二维 Z 网格，shape=(n_thr, n_thr)
      signal_counts:   二维 S 网格
      background_counts: 二维 B 网格
      best_threshold:  (thr_vvv, thr_vh) 二元组
      其余 *_errors、*_entries 同理为二维；best_* 为标量
    """
    # 1) 权重转换（保持完全不变）
    w = convert_weights(w_old, y_test, lumi, channel, test_ratio, isMulti=True)

    # 2) 多分类打分：列顺序假定为 0=BKG, 1=VVV, 2=VH（与你的训练保持一致）
    proba = model.predict_proba(X_test)
    p_bkg = proba[:, 0]
    p_vvv = proba[:, 1]
    p_vh  = proba[:, 2]

    # 3) 二维阈值扫描网格（与原单轴一致的幂变换分布，靠近1更密集）
    p_pow = 0.01
    x = np.linspace(0, 1, n_thresholds)
    thresholds = x ** p_pow  # 一维阈值数组
    nT = thresholds.size

    # 结果网格
    S_grid   = np.zeros((nT, nT), dtype=float)
    B_grid   = np.zeros((nT, nT), dtype=float)
    S_ent    = np.zeros((nT, nT), dtype=int)
    B_ent    = np.zeros((nT, nT), dtype=int)
    sS_grid  = np.zeros((nT, nT), dtype=float)  # sigma_S
    sB_grid  = np.zeros((nT, nT), dtype=float)  # sigma_B
    Z_grid   = np.zeros((nT, nT), dtype=float)
    sZ_grid  = np.zeros((nT, nT), dtype=float)

    # 真值掩码：多分类
    is_sig_true = (y_test == 1) | (y_test == 2)  # VVV 或 VH
    is_bkg_true = (y_test == 0)

    # 4) 二维扫描
    for i, thr_vvv in enumerate(thresholds):
        mask_vvv = (p_vvv >= thr_vvv)
        for j, thr_vh in enumerate(thresholds):
            mask_vh  = (p_vh  >= thr_vh)
            # 通过选择的事件：VVV 或 VH 有其一过阈值
            mask_sel = mask_vvv | mask_vh

            # S/B（按真值）
            mS = is_sig_true & mask_sel
            mB = is_bkg_true & mask_sel

            wS = w[mS]
            wB = w[mB]

            S  = wS.sum()
            B  = wB.sum()
            Se = int(mS.sum())
            Be = int(mB.sum())

            sigma_S = np.sqrt((wS ** 2).sum()) if Se > 0 else 0.0
            sigma_B = np.sqrt((wB ** 2).sum()) if Be > 0 else 0.0

            if (S + B) > 0.0:
                Z = S / np.sqrt(S + B)
                denom32 = 2.0 * (S + B) ** 1.5
                dZ_dS = (S + 2.0 * B) / denom32
                dZ_dB = -S / denom32
                sigma_Z = np.sqrt((dZ_dS * sigma_S) ** 2 +
                                  (dZ_dB * sigma_B) ** 2)
            else:
                Z = 0.0
                sigma_Z = 0.0

            S_grid[i, j]  = S
            B_grid[i, j]  = B
            S_ent[i, j]   = Se
            B_ent[i, j]   = Be
            sS_grid[i, j] = sigma_S
            sB_grid[i, j] = sigma_B
            Z_grid[i, j]  = Z
            sZ_grid[i, j] = sigma_Z

    # 5) 最优点（二维）
    idx_flat = int(np.nanargmax(Z_grid))
    i_opt, j_opt = np.unravel_index(idx_flat, Z_grid.shape)
    thr_vvv_best = thresholds[i_opt]
    thr_vh_best  = thresholds[j_opt]

    # 最优掩码（用于后续分类别显著度与绘图）
    mask_pass = (p_vvv >= thr_vvv_best) | (p_vh >= thr_vh_best)

    # 6) 分类别显著度（沿用原来的类别区分与 SCALE/WEIGHT 方式，不改变）
    category_names = ['www', 'wwz', 'wzz', 'zzz', 'wh', 'zh']  # 保持不变
    cat_Z, cat_dZ = [], []
    cat_S, cat_B = [], []

    WEIGHT = WEIGHT_2FAT if channel == 2 else WEIGHT_3FAT

    for name in category_names:
        if name == 'wh':
            SCALE = SCALE_2FAT_VH if channel == 2 else SCALE_3FAT_VH
            mask_cat_old = (np.isclose(w_old, WEIGHT['wplush'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wminush'] * SCALE))
        elif name == 'zh':
            SCALE = SCALE_2FAT_VH if channel == 2 else SCALE_3FAT_VH
            mask_cat_old = np.isclose(w_old, WEIGHT['zh'] * SCALE)
        else:
            SCALE = SCALE_2FAT_VVV if channel == 2 else SCALE_3FAT_VVV
            mask_cat_old = np.isclose(w_old, WEIGHT[name.lower()] * SCALE)

        # 信号：属于该类别 且 通过选择
        mask_Sj = mask_cat_old & mask_pass
        # 本底：通过选择但不属于该类别
        mask_Bj = mask_pass & ~mask_cat_old

        w_Sj = w[mask_Sj];   w_Bj = w[mask_Bj]
        Sj   = w_Sj.sum();   Bj   = w_Bj.sum()
        sigma_Sj = np.sqrt((w_Sj**2).sum()) if mask_Sj.any() else 0.0
        sigma_Bj = np.sqrt((w_Bj**2).sum()) if mask_Bj.any() else 0.0

        if Sj + Bj > 0:
            Zj = Sj / np.sqrt(Sj + Bj)
            denom32_j = 2.0 * (Sj + Bj)**1.5
            dZ_dSj = (Sj + 2 * Bj) / denom32_j
            dZ_dBj = -Sj / denom32_j
            sigma_Zj = np.sqrt((dZ_dSj * sigma_Sj)**2 +
                               (dZ_dBj * sigma_Bj)**2)
        else:
            Zj, sigma_Zj = 0.0, 0.0

        cat_Z.append(Zj)
        cat_dZ.append(sigma_Zj)
        cat_S.append(Sj)
        cat_B.append(Bj)

    # 7) 绘图（保持风格；分数用 max(pVVV,pVH)）
    plot_masks = {}

    # 先加入原有 category_names
    category_names = ['vvv', 'vh']
    for name in category_names:
        if name == 'vh':
            SCALE = SCALE_2FAT_VH if channel == 2 else SCALE_3FAT_VH
            mask_cat_old = (np.isclose(w_old, WEIGHT['wplush'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wminush'] * SCALE) |
                            np.isclose(w_old, WEIGHT['zh'] * SCALE))
        elif name == 'vvv':
            SCALE = SCALE_2FAT_VH if channel == 2 else SCALE_3FAT_VH
            mask_cat_old = (np.isclose(w_old, WEIGHT['www'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wwz'] * SCALE) |
                            np.isclose(w_old, WEIGHT['wzz'] * SCALE) |
                            np.isclose(w_old, WEIGHT['zzz'] * SCALE))
        plot_masks[name] = mask_cat_old

    # 再加入 qcd / vv / ttbar（保持不变）
    plot_masks['qcd'] = (
        np.isclose(w_old, WEIGHT['qcd_ht100to200']) |
        np.isclose(w_old, WEIGHT['qcd_ht200to400']) |
        np.isclose(w_old, WEIGHT['qcd_ht400to600']) |
        np.isclose(w_old, WEIGHT['qcd_ht600to800']) |
        np.isclose(w_old, WEIGHT['qcd_ht800to1000']) |
        np.isclose(w_old, WEIGHT['qcd_ht1000to1200']) |
        np.isclose(w_old, WEIGHT['qcd_ht1200to1500']) |
        np.isclose(w_old, WEIGHT['qcd_ht1500to2000']) |
        np.isclose(w_old, WEIGHT['qcd_ht2000toinf'])
    )
    plot_masks['vv'] = (
        np.isclose(w_old, WEIGHT['ww']) |
        np.isclose(w_old, WEIGHT['wz']) |
        np.isclose(w_old, WEIGHT['zz'])
    )
    plot_masks['ttbar'] = (
        np.isclose(w_old, WEIGHT['tt_had']) |
        np.isclose(w_old, WEIGHT['tt_semilep'])
    )

    bins = np.linspace(0., 1, 51)
    plt.figure()
    for label, m in plot_masks.items():
        if np.any(m):
            plt.hist(p_vvv[m], bins=bins, range=(0, 1),
                     weights=w[m],
                     histtype='step', linewidth=2, label=label)
    plt.xlim(0, 1)
    plt.xlabel("VVV Score")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.ylim(1, None)
    plt.legend(ncol=1, fontsize=24)
    plt.tight_layout()
    plt.show()

    plt.figure()
    for label, m in plot_masks.items():
        if np.any(m):
            plt.hist(p_vh[m], bins=bins, range=(0, 1),
                     weights=w[m],
                     histtype='step', linewidth=2, label=label)
    plt.xlim(0, 1)
    plt.xlabel("VH Score")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.ylim(1, None)
    plt.legend(ncol=1, fontsize=24)
    plt.tight_layout()
    plt.show()

    bins = np.linspace(0., 0.05, 51)
    plt.figure()
    for label, m in plot_masks.items():
        if np.any(m):
            plt.hist(p_vvv[m], bins=bins, range=(0., 0.05),
                     weights=w[m],
                     histtype='step', linewidth=2, label=label)
    plt.xlim(0., 0.05)
    plt.xlabel("VVV Score")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.ylim(1, None)
    plt.legend(ncol=2, fontsize=24)
    plt.tight_layout()
    plt.show()

    plt.figure()
    for label, m in plot_masks.items():
        if np.any(m):
            plt.hist(p_vh[m], bins=bins, range=(0., 0.05),
                     weights=w[m],
                     histtype='step', linewidth=2, label=label)
    plt.xlim(0., 0.05)
    plt.xlabel("VH Score")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.ylim(1, None)
    plt.legend(ncol=2, fontsize=24)
    plt.tight_layout()
    plt.show()

    # 8) 效率（按真值）
    S_total = w[is_sig_true].sum()
    B_total = w[is_bkg_true].sum()
    best_S  = S_grid[i_opt, j_opt]
    best_B  = B_grid[i_opt, j_opt]
    best_signal_efficiency     = best_S / S_total if S_total > 0 else np.nan
    best_background_efficiency = best_B / B_total if B_total > 0 else np.nan
    category_names = ['www', 'wwz', 'wzz', 'zzz', 'wh', 'zh']

    return {
        'thresholds':                 thresholds,           # 一维
        'significances':              Z_grid,               # 二维
        'significance_errors':        sZ_grid,              # 二维
        'signal_counts':              S_grid,               # 二维
        'background_counts':          B_grid,               # 二维
        'signal_errors':              sS_grid,              # 二维
        'background_errors':          sB_grid,              # 二维
        'signal_entries':             S_ent,                # 二维
        'background_entries':         B_ent,                # 二维
        'best_threshold':             (thr_vvv_best, thr_vh_best),  # 二元组
        'best_significance':          Z_grid[i_opt, j_opt],
        'best_significance_error':    sZ_grid[i_opt, j_opt],
        'best_S':                     best_S,
        'best_B':                     best_B,
        'best_S_entry':               S_ent[i_opt, j_opt],
        'best_B_entry':               B_ent[i_opt, j_opt],
        'category_best_S':        np.array(cat_S),
        'category_best_B':        np.array(cat_B),
        'category_names':             category_names,
        'category_significances':     np.array(cat_Z),
        'category_significance_errors': np.array(cat_dZ),
        'best_signal_efficiency':     best_signal_efficiency,
        'best_background_efficiency': best_background_efficiency,
    }

In [ ]:
def plot_significance_curve(thresholds: np.ndarray,
                            significances: np.ndarray,
                            isMulti: bool = False,
                            best_threshold=None):
    if not isMulti:
        # —— 原 1D 版本，保持不变 ——
        plt.figure(figsize=(6, 4))
        plt.plot(thresholds, significances, marker=None, linestyle='-')
        plt.xlabel("BDT Score")
        plt.ylabel(r"$\frac{S}{\sqrt{S+B}}$")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()
        return

    # —— 2D 版本（多分类适配）：significances 为 (n, n)；thresholds 为 1D 长度 n ——
    Z = np.asarray(significances)
    thr = np.asarray(thresholds)
    assert Z.ndim == 2 and Z.shape[0] == Z.shape[1] == thr.size, \
        "significances 应为 (n,n)，且与 thresholds 长度一致"

    n = thr.size

    # 大网格下采样仅用于绘图（不影响 Z 的寻优等计算结果）
    # 目标：最长不超过 ~400 个像素格，避免 10k x 10k 图像卡死
    max_bins = 400
    step = max(1, int(np.ceil(n / max_bins)))
    thr_plot = thr[::step]
    Z_plot = Z[::step, ::step]

    # 把“中心点”转为“边界”用于 pcolormesh 显示（非均匀 bin）
    def _edges_from_centers(c):
        e = np.empty(c.size + 1, dtype=float)
        e[1:-1] = 0.5 * (c[1:] + c[:-1])
        # 线性外推
        e[0]  = c[0]  - (e[1]  - c[0])
        e[-1] = c[-1] + (c[-1] - e[-2])
        # 限制在 [0,1]（本问题阈值域）
        return np.clip(e, 0.0, 1.0)

    xedges = _edges_from_centers(thr_plot)  # x: thr_vvv
    yedges = _edges_from_centers(thr_plot)  # y: thr_vh（与 x 共享同一阈值序列）

    # 注意：扫描时我们保存的是 Z[i_vvv, j_vh]，而 pcolormesh 需要 Z[y, x]
    # 因此需要转置，使得 x=VVV, y=VH
    Z_disp = Z_plot.T

    plt.figure(figsize=(7, 6))
    pcm = plt.pcolormesh(xedges, yedges, Z_disp, shading='auto')
    cbar = plt.colorbar(pcm)
    cbar.set_label(r"$\frac{S}{\sqrt{S+B}}$")

    # 叠加最优点
    if best_threshold is not None and isinstance(best_threshold, (tuple, list, np.ndarray)) and len(best_threshold) == 2:
        plt.plot(best_threshold[0], best_threshold[1], marker='+', markersize=12, mew=2, linestyle='None', label='Best')

    plt.xlabel(r"VVV score threshold")
    plt.ylabel(r"VH score threshold")
    ###plt.title("Significance vs thresholds (VVV, VH)")
    if best_threshold is not None:
        plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
def _fmt_nd(val, fmt="{:.6f}"):
    """把标量或数组统一格式化为字符串；数组展平成 [v1, v2, ...]"""
    try:
        if np.isscalar(val):
            return fmt.format(val)
        arr = np.asarray(val).ravel()
        return "[" + ", ".join(fmt.format(x) for x in arr) + "]"
    except Exception:
        return str(val)

In [ ]:
def get_weighted_model(X, y, w, model_name):

    # === 1) 根据模型名判断当时去相关的特征列 ===
    # 注意：这里要和 train_multi_model_un 里 decorrelate_feature_names 对应的列保持一致
    decor_cols = []
    if "fat2" in model_name:
        # 训练 bdt_5class_with_tagger_fat2_nocorr.pkl 时 decorrelate 的是 sphereM
        decor_cols = ["msd8_1"]
    elif "fat3" in model_name:
        # 如果你对 fat3 也是用 msd8_1/2/3 去相关，就写在这里
        decor_cols = ["msd8_1"]

    # === 2) 从 X 中删去这些 decor 列，得到和训练时相同维度的特征 ===
    if hasattr(X, "columns"):  # pandas DataFrame
        # 只删确实存在的列，避免 key error
        drop_cols = [c for c in decor_cols if c in X.columns]
        X_used = X.drop(columns=drop_cols)
    else:  # numpy array，需要通过 FEATURE_NAMES 来找索引
        name_to_idx = {name: i for i, name in enumerate(FEATURE_NAMES)}
        drop_idx = [name_to_idx[c] for c in decor_cols if c in name_to_idx]
        keep_idx = [i for i in range(X.shape[1]) if i not in drop_idx]
        X_used = X[:, keep_idx]

    # 3) 划分 train vs test（注意用 X_used 而不是原始 X）
    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X_used, y, w,
        test_size=0.3,
        stratify=y,
        random_state=RANDOM_STATE
    )

    model_path = os.path.join("python", model_name)
    ext = os.path.splitext(model_path)[1].lower()

    class _BoosterWrapper:
        """
        兜底：当 json/ubj 只能用 xgboost.Booster 载入时，包一层使其具备 predict_proba 接口，
        并在需要时对 raw margin 做 sigmoid/softmax 转换。
        """
        def __init__(self, booster):
            self._booster = booster
            try:
                self.n_features_in_ = int(booster.num_features())
            except Exception:
                pass

        def get_booster(self):
            return self._booster

        @staticmethod
        def _sigmoid(x):
            x = np.asarray(x)
            return 1.0 / (1.0 + np.exp(-x))

        @staticmethod
        def _softmax(m):
            m = np.asarray(m)
            m = m - np.max(m, axis=1, keepdims=True)
            e = np.exp(m)
            return e / np.sum(e, axis=1, keepdims=True)

        def predict_proba(self, X):
            import xgboost as xgb
            dmat = xgb.DMatrix(X)
            pred = self._booster.predict(dmat)  # 可能是 prob，也可能是 raw margin

            pred = np.asarray(pred)
            if pred.ndim == 1:
                # binary: pred 可能是 prob 或 margin
                if pred.min() < 0.0 or pred.max() > 1.0:
                    p1 = self._sigmoid(pred)
                else:
                    p1 = pred
                return np.vstack([1.0 - p1, p1]).T
            else:
                # multiclass: pred 可能是 softprob 或 raw margin
                row_sum = pred.sum(axis=1)
                if (pred.min() < 0.0) or (pred.max() > 1.0) or (not np.allclose(row_sum, 1.0, rtol=1e-3, atol=1e-3)):
                    pred = self._softmax(pred)
                return pred

        def predict(self, X):
            proba = self.predict_proba(X)
            return np.argmax(proba, axis=1)

    if ext in [".pkl", ".pickle"]:
        with open(model_path, "rb") as f:
            print(f)
            clf = pickle.load(f)

    elif ext in [".json", ".ubj"]:
        # 优先尝试 sklearn wrapper（保持和你原来 clf 的接口一致）
        try:
            import xgboost as xgb
            clf = xgb.XGBClassifier()
            clf.load_model(model_path)
        except Exception:
            # 兜底：直接 Booster 载入，再包一层 predict_proba
            import xgboost as xgb
            booster = xgb.Booster()
            booster.load_model(model_path)
            clf = _BoosterWrapper(booster)

    else:
        # 兜底：先按 pkl 读，失败再按 xgboost 模型读
        try:
            with open(model_path, "rb") as f:
                print(f)
                clf = pickle.load(f)
        except Exception:
            import xgboost as xgb
            try:
                clf = xgb.XGBClassifier()
                clf.load_model(model_path)
            except Exception:
                booster = xgb.Booster()
                booster.load_model(model_path)
                clf = _BoosterWrapper(booster)

    # （可选）安全检查：特征数要和模型一致
    if hasattr(clf, "n_features_in_"):
        if clf.n_features_in_ != X_train.shape[1]:
            raise ValueError(
                f"Model expects {clf.n_features_in_} features, "
                f"but X has {X_train.shape[1]} after dropping decor cols {decor_cols}."
            )
    elif hasattr(clf, "get_booster"):
        try:
            nfeat = int(clf.get_booster().num_features())
            if nfeat != X_train.shape[1]:
                raise ValueError(
                    f"Model expects {nfeat} features (from booster), "
                    f"but X has {X_train.shape[1]} after dropping decor cols {decor_cols}."
                )
        except Exception:
            pass

    return clf, (X_train, X_test,
                 y_train, y_test,
                 w_train, w_test)

#ix1 = X2_std.columns.get_loc("sphereM")
#print(X2_std)
clf2, splits2 = get_weighted_model(X2_std, y2, w2, 'bdt_fat2_nocorr_v3.json')

result = find_optimal_significance_combine(
    clf2, splits2[1], splits2[3], splits2[5],
    lumi=171.0,
    channel=2,
    test_ratio=0.3,
    n_thresholds=10,
    bkgRatio=1,
    topN=4,                 # 可调
    min_bkg_weight=5.0,
    nClass=5,
    debug=False,         # 开启详细打印
    #debug_topk=1,      # 每步打印前10个组合
)

for b in result['top_bins']:
    thr_low_str  = _fmt_nd(b['thr_low'])
    thr_high_str = _fmt_nd(b['thr_high'])
    tail_sig_eff_str = _fmt_nd(b['tail_signal_efficiency'])
    tail_bkg_eff_str = _fmt_nd(b['tail_background_efficiency'])

    print(f"\n-- Bin {b['bin_index']} --")
    print(f"区间: [thr_low={thr_low_str}, thr_high={thr_high_str})")
    print(f"显著度: {b['significance']:.3f} ± {b['significance_error']:.3f}")
    print(f"S(加权): {b['S']:.3f} ± {b['S_err']:.3f}  |  S(条目): {b['S_entries']}")
    print(f"B(加权): {b['B']:.3f} ± {b['B_err']:.3f}  |  B(条目): {b['B_entries']}")
    print(f"bin 内信号效率: {b['bin_signal_efficiency']:.6f}  |  bin 内本底效率: {b['bin_background_efficiency']:.6f}")
    print(f"(低阈值)尾部信号效率: {tail_sig_eff_str}  |  (低阈值)尾部本底效率: {tail_bkg_eff_str}")

    print("  · 各子信号显著度：")
    for name, Zj, dZj in zip(b['category_names'], b['category_significances'], b['category_significance_errors']):
        print(f"    {name}: {Zj:.3f} ± {dZj:.3f}")

    print("  · 子信号计数分解（加权±误差）：")
    for name, Sj, dSj, Bj, dBj in zip(b['category_names'], b['category_S'], b['category_S_err'], b['category_B'], b['category_B_err']):
        print(f"    {name}: S={Sj:.3f} ± {dSj:.3f} | B(rest)={Bj:.3f} ± {dBj:.3f}")

    print("  · 本底大类分解（加权±误差）：")
    for name, Bj, dBj in zip(b['bkg_category_names'], b['bkg_B'], b['bkg_B_err']):
        print(f"    {name}: B={Bj:.3f} ± {dBj:.3f}")

# === 合并前 N 个 bin 的总体显著度 ===
print("\n==== 前 N 个 bin 合并后的总体 ====")
print("S(加权)合计: {:.3f} ± {:.3f}".format(result['combined_topN_S'], result['combined_topN_S_err']))
print("B(加权)合计: {:.3f} ± {:.3f}".format(result['combined_topN_B'], result['combined_topN_B_err']))
print("合并显著度: {:.3f} ± {:.3f}".format(result['combined_topN_significance'], result['combined_topN_significance_error']))

# 保留原有曲线绘图（只在 VVV 轴）
#plot_significance_curve(result['thresholds'], result['significances'])


In [ ]:
#clf3, splits3 = get_weighted_model(X_cr13, y_cr13, w_cr13, 'bdt_weighted_with_tagger_fat3.pkl')

result = find_optimal_significance_combine(
    clf3, splits3[1], splits3[3], splits3[5],
    lumi=171.0,
    channel=3,
    test_ratio=0.3,
    n_thresholds=10,
    bkgRatio=1,
    topN=4,                 # 可调
    min_bkg_weight=5.0,     # 可调
    nClass=5
)

for b in result['top_bins']:
    thr_low_str  = _fmt_nd(b['thr_low'])
    thr_high_str = _fmt_nd(b['thr_high'])
    tail_sig_eff_str = _fmt_nd(b['tail_signal_efficiency'])
    tail_bkg_eff_str = _fmt_nd(b['tail_background_efficiency'])

    print(f"\n-- Bin {b['bin_index']} --")
    print(f"区间: [thr_low={thr_low_str}, thr_high={thr_high_str})")
    print(f"显著度: {b['significance']:.3f} ± {b['significance_error']:.3f}")
    print(f"S(加权): {b['S']:.3f} ± {b['S_err']:.3f}  |  S(条目): {b['S_entries']}")
    print(f"B(加权): {b['B']:.3f} ± {b['B_err']:.3f}  |  B(条目): {b['B_entries']}")
    print(f"bin 内信号效率: {b['bin_signal_efficiency']:.6f}  |  bin 内本底效率: {b['bin_background_efficiency']:.6f}")
    print(f"(低阈值)尾部信号效率: {tail_sig_eff_str}  |  (低阈值)尾部本底效率: {tail_bkg_eff_str}")

    print("  · 各子信号显著度：")
    for name, Zj, dZj in zip(b['category_names'], b['category_significances'], b['category_significance_errors']):
        print(f"    {name}: {Zj:.3f} ± {dZj:.3f}")

    print("  · 子信号计数分解（加权±误差）：")
    for name, Sj, dSj, Bj, dBj in zip(b['category_names'], b['category_S'], b['category_S_err'], b['category_B'], b['category_B_err']):
        print(f"    {name}: S={Sj:.3f} ± {dSj:.3f} | B(rest)={Bj:.3f} ± {dBj:.3f}")

    print("  · 本底大类分解（加权±误差）：")
    for name, Bj, dBj in zip(b['bkg_category_names'], b['bkg_B'], b['bkg_B_err']):
        print(f"    {name}: B={Bj:.3f} ± {dBj:.3f}")

# === 合并前 N 个 bin 的总体显著度 ===
print("\n==== 前 N 个 bin 合并后的总体 ====")
print("S(加权)合计: {:.3f} ± {:.3f}".format(result['combined_topN_S'], result['combined_topN_S_err']))
print("B(加权)合计: {:.3f} ± {:.3f}".format(result['combined_topN_B'], result['combined_topN_B_err']))
print("合并显著度: {:.3f} ± {:.3f}".format(result['combined_topN_significance'], result['combined_topN_significance_error']))

# 保留原有曲线绘图
#plot_significance_curve(result['thresholds'], result['significances'])

In [ ]:
def get_weighted_model(X, y, w, model_name):

    # === 1) 根据模型名判断当时去相关的特征列 ===
    # 注意：这里要和 train_multi_model_un 里 decorrelate_feature_names 对应的列保持一致
    decor_cols = []
    #if "fat2" in model_name:
    #    # 训练 bdt_5class_with_tagger_fat2_nocorr.pkl 时 decorrelate 的是 sphereM
    #    decor_cols = ["msd8_1"]
    #elif "fat3" in model_name:
    #    # 如果你对 fat3 也是用 msd8_1/2/3 去相关，就写在这里
    #    decor_cols = ["msd8_1"]

    # === 2) 从 X 中删去这些 decor 列，得到和训练时相同维度的特征 ===
    if hasattr(X, "columns"):  # pandas DataFrame
        # 只删确实存在的列，避免 key error
        drop_cols = [c for c in decor_cols if c in X.columns]
        X_used = X.drop(columns=drop_cols)
    else:  # numpy array，需要通过 FEATURE_NAMES 来找索引
        name_to_idx = {name: i for i, name in enumerate(FEATURE_NAMES)}
        drop_idx = [name_to_idx[c] for c in decor_cols if c in name_to_idx]
        keep_idx = [i for i in range(X.shape[1]) if i not in drop_idx]
        X_used = X[:, keep_idx]

    # 3) 划分 train vs test（注意用 X_used 而不是原始 X）
    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X_used, y, w,
        test_size=0.3,
        stratify=y,
        random_state=RANDOM_STATE
    )

    # 4) 读取已经训练好的模型（兼容 .pkl/.pickle 和 .json/.ubj）
    import os
    import pickle
    import numpy as np

    model_path = os.path.join("python", model_name)
    ext = os.path.splitext(model_path)[1].lower()

    class _BoosterWrapper:
        """
        兜底：当 json/ubj 只能用 xgboost.Booster 载入时，包一层使其具备 predict_proba 接口，
        并在需要时对 raw margin 做 sigmoid/softmax 转换。
        """
        def __init__(self, booster):
            self._booster = booster
            try:
                self.n_features_in_ = int(booster.num_features())
            except Exception:
                pass

        def get_booster(self):
            return self._booster

        @staticmethod
        def _sigmoid(x):
            x = np.asarray(x)
            return 1.0 / (1.0 + np.exp(-x))

        @staticmethod
        def _softmax(m):
            m = np.asarray(m)
            m = m - np.max(m, axis=1, keepdims=True)
            e = np.exp(m)
            return e / np.sum(e, axis=1, keepdims=True)

        def predict_proba(self, X):
            import xgboost as xgb
            dmat = xgb.DMatrix(X)
            pred = self._booster.predict(dmat)  # 可能是 prob，也可能是 raw margin

            pred = np.asarray(pred)
            if pred.ndim == 1:
                # binary: pred 可能是 prob 或 margin
                if pred.min() < 0.0 or pred.max() > 1.0:
                    p1 = self._sigmoid(pred)
                else:
                    p1 = pred
                return np.vstack([1.0 - p1, p1]).T
            else:
                # multiclass: pred 可能是 softprob 或 raw margin
                row_sum = pred.sum(axis=1)
                if (pred.min() < 0.0) or (pred.max() > 1.0) or (not np.allclose(row_sum, 1.0, rtol=1e-3, atol=1e-3)):
                    pred = self._softmax(pred)
                return pred

        def predict(self, X):
            proba = self.predict_proba(X)
            return np.argmax(proba, axis=1)

    if ext in [".pkl", ".pickle"]:
        with open(model_path, "rb") as f:
            print(f)
            clf = pickle.load(f)

    elif ext in [".json", ".ubj"]:
        # 优先尝试 sklearn wrapper（保持和你原来 clf 的接口一致）
        try:
            import xgboost as xgb
            clf = xgb.XGBClassifier()
            clf.load_model(model_path)
        except Exception:
            # 兜底：直接 Booster 载入，再包一层 predict_proba
            import xgboost as xgb
            booster = xgb.Booster()
            booster.load_model(model_path)
            clf = _BoosterWrapper(booster)

    else:
        # 兜底：先按 pkl 读，失败再按 xgboost 模型读
        try:
            with open(model_path, "rb") as f:
                print(f)
                clf = pickle.load(f)
        except Exception:
            import xgboost as xgb
            try:
                clf = xgb.XGBClassifier()
                clf.load_model(model_path)
            except Exception:
                booster = xgb.Booster()
                booster.load_model(model_path)
                clf = _BoosterWrapper(booster)

    # （可选）安全检查：特征数要和模型一致
    '''
    if hasattr(clf, "n_features_in_"):
        if clf.n_features_in_ != X_train.shape[1]:
            raise ValueError(
                f"Model expects {clf.n_features_in_} features, "
                f"but X has {X_train.shape[1]} after dropping decor cols {decor_cols}."
            )
    elif hasattr(clf, "get_booster"):
        try:
            nfeat = int(clf.get_booster().num_features())
            if nfeat != X_train.shape[1]:
                raise ValueError(
                    f"Model expects {nfeat} features (from booster), "
                    f"but X has {X_train.shape[1]} after dropping decor cols {decor_cols}."
                )
        except Exception:
            pass
    '''

    return clf, (X_train, X_test,
                 y_train, y_test,
                 w_train, w_test)

#ix1 = X2_std.columns.get_loc("sphereM")
#print(X2_std)
clf2, splits2 = get_weighted_model(X2_std, y2, w2, 'bdt_fat2_nocorr_v3.json')

In [ ]:
SCALE_2FAT_VVV= 1875.150046658813
SCALE_2FAT_VH= 4019.6688183605593
SCALE_2FAT_VV= 60.90874629593156
SCALE_2FAT_TT= 5.299021879414808
SCALE_2FAT_QCD= 0.0025184026063541126

SCALE_3FAT_VVV= 7923.170542833438
SCALE_3FAT_VH= 37807.21909945096
SCALE_3FAT_VV= 651.3756086212677
SCALE_3FAT_TT= 11.443017490367913
SCALE_3FAT_QCD= 0.047065666680667216

In [ ]:
def plot_multi_results_un_sel(clf, splits, tree_name, decorrelate_feature_names=None):
    """
    适配 5-class BDT 的可视化：
      (1) ROC：VVV 与其余 4 类分别的 ROC；VH 与其余 4 类分别的 ROC。
      (2) 特征重要性：用真实特征名标注（与训练输入一致），x 轴 log。
      (3) Score 分布：VVV vs rest、VH vs rest（one-vs-rest）。
      (4) Loss 曲线、特征相关矩阵（基于训练输入）。
      (5) 去相关检验（仅此处使用“包含 decor 特征”的 X）：
          5.1 相关系数热力图：各类概率分数 vs 指定 decor 变量（Train/Test）
          5.2 散点图：指定 decor 变量 vs VVV 分数（逐变量一张）
    """

    X_train_full, X_test_full, y_train, y_test, w_train, w_test = splits

    # ====== 0) 取得“全特征名”与 decor 索引；构造“训练使用的特征名/矩阵” ======
    # 优先用 DataFrame 的列名；没有则退回你预置的 branches*；再不行用 f{i}
    def _get_full_feature_names(X_like):
        if hasattr(X_like, "columns"):
            return list(X_like.columns)
        if tree_name == "fat2":
            return list(branches2)
        elif tree_name == "fat3":
            return list(branches3)
        # fallback
        ncols = X_like.shape[1]
        return [f"f{i}" for i in range(ncols)]

    full_feature_names = _get_full_feature_names(X_train_full)

    # 将 decor 名字/索引解析到 full_feature_names 的列号
    def _resolve_indices(names_or_idx, feature_names_list):
        if not names_or_idx:
            return []
        name_to_idx = {c: i for i, c in enumerate(feature_names_list)}
        out = []
        for key in names_or_idx:
            if isinstance(key, int):
                if 0 <= key < len(feature_names_list):
                    out.append(key)
                else:
                    print(f"[WARN] decor 列号 {key} 越界，已跳过。")
            else:
                if key in name_to_idx:
                    out.append(name_to_idx[key])
                else:
                    print(f"[INFO] decor 变量 '{key}' 不在全特征列表中，跳过。")
        # 去重保序
        seen, res = set(), []
        for i in out:
            if i not in seen:
                seen.add(i); res.append(i)
        return res

    decor_idx_full = _resolve_indices(decorrelate_feature_names, full_feature_names)
    all_idx = np.arange(len(full_feature_names))
    keep_idx = np.setdiff1d(all_idx, np.array(decor_idx_full, dtype=int))  # 训练真正使用的列

    # 构造“用于预测/绘图（除去 decor）”的 X，以及“仅供 decor 检验（包含 decor）”的 X_full
    def _slice_cols(X_like, idx):
        if hasattr(X_like, "iloc"):
            return X_like.iloc[:, idx]
        return X_like[:, idx]

    X_train_used = _slice_cols(X_train_full, keep_idx)
    X_test_used  = _slice_cols(X_test_full,  keep_idx)

    # 训练时使用的特征名（对应 clf.n_features_in_ 的顺序）
    feat_names_used = [full_feature_names[i] for i in keep_idx]

    # 与 clf 期望的特征数做一次对齐校验；不符则尝试退回
    if hasattr(clf, "n_features_in_") and getattr(clf, "n_features_in_") != X_train_used.shape[1]:
        # 如果 full 就匹配 clf，则认为 splits 已经是“已剔除 decor”的版本
        if hasattr(X_train_full, "shape") and X_train_full.shape[1] == getattr(clf, "n_features_in_"):
            X_train_used, X_test_used = X_train_full, X_test_full
            feat_names_used = full_feature_names
            decor_idx_full = _resolve_indices([], full_feature_names)  # decor 检验将被跳过
            print("[INFO] 检测到 splits 已与模型输入对齐，按原 X 使用。")
        else:
            raise ValueError(
                f"特征数与模型不一致：X_used={X_train_used.shape[1]} vs clf.n_features_in_={getattr(clf,'n_features_in_',None)}"
            )

    # ====== 1) 常量与工具 ======
    IDX_VVV, IDX_VH, IDX_VV, IDX_TT, IDX_QCD = 0, 1, 2, 3, 4
    class_names = ["VVV", "VH", "TT", "VV", "QCD"]
    palette = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple']

    probs_train = clf.predict_proba(X_train_used)
    probs_test  = clf.predict_proba(X_test_used)

    try:

        if decor_idx_full is None:
            print("[INFO] 未在特征列表中找到 sphereM，跳过 sphereM 分布绘图。")
        else:
            # 统一将 X_test_full 转成 ndarray，便于按列切片
            if hasattr(X_test_full, "iloc"):
                X_test_arr = X_test_full.to_numpy()
            else:
                X_test_arr = np.asarray(X_test_full)

            sphereM_all = X_test_arr[:, decor_idx_full]
            y_all = y_test
            w_all = w_test

            # VVV / VH 两个 BDT score 通道
            bdt_channels = [
                (IDX_VVV, "VVV score"),
                (IDX_VH,  "VH score"),
            ]
            thresholds = [0.0, 0.2, 0.5, 0.9, 0.95]

            # 颜色：沿用当前 CMS 样式的前 5 种
            try:
                color_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", [])
            except Exception:
                color_cycle = []
            if len(color_cycle) < len(thresholds):
                # 退回到 Matplotlib 默认颜色
                color_cycle = ["C0", "C1", "C2", "C3", "C4"]
            colors = color_cycle[:len(thresholds)]

            # 对每个物理类别分别画图
            for class_idx, class_name in enumerate(class_names):
                # 先确定该类在 Test 集中的 sphereM 范围，以保证不同 cut 使用相同 bin
                mask_cls = (y_all == class_idx)
                if not np.any(mask_cls):
                    continue
                sphereM_cls = sphereM_all[mask_cls]

                # 若数值全部相同，扩一点范围避免 bin 出问题
                xmin = np.min(sphereM_cls)
                xmax = np.max(sphereM_cls)
                if not np.isfinite(xmin) or not np.isfinite(xmax):
                    continue
                if xmin == xmax:
                    xmin -= 1.0
                    xmax += 1.0
                bin_edges = np.linspace(-2.5, 2, 51)

                for bdt_idx, bdt_name in bdt_channels:
                    scores = probs_test[:, bdt_idx]

                    plt.figure(figsize=(8, 6))
                    for thr, c in zip(thresholds, colors):
                        sel = mask_cls & (scores > thr)
                        if not np.any(sel):
                            continue
                        sm_sel = sphereM_all[sel]
                        w_sel = w_all[sel]

                        plt.hist(
                            sm_sel,
                            bins=bin_edges,
                            weights=w_sel,
                            density=True,
                            histtype="step",
                            linewidth=2.0,
                            label=f"{bdt_name} > {thr:g}",
                            color=c,
                        )

                    plt.xlabel("sphereM")
                    plt.ylabel("Density")
                    ###plt.title(
                        f"{tree_name} sphereM distribution for {class_name} "
                        f"(Test, selected by {bdt_name})",
                        fontsize=14,
                    )
                    plt.legend()
                    plt.tight_layout()
                    plt.show()
    except Exception as e:
        print(f"[WARN] sphereM 分布绘图时发生异常: {e}")



In [ ]:
plot_multi_results_un_sel(clf2, splits2, 'fat2', decorrelate_feature_names=["sphereM"])